# 🎬 Cultural Screenplay & Visual Adaptation Studio (Sanskritik Adaptation Agent)

**Uddeshya:** Ek screenplay ko ek *exact* Indian culture + dialect (jaise Majhi, Malwai, Bangru, Marwari, Mewari) mein *re-create* karna — sirf translate nahi — aur uske baad ek **consistent visual production pack** banana (character bible, costume bible, scene keyframes).
# Quick Explanation (Short)

This notebook is an **AI-powered cultural screenplay adaptation pipeline**. It takes a screenplay, identifies characters/scenes/props/costumes, normalizes them into canonical IDs, checks continuity, creates a culture-specific adaptation plan, rewrites the screenplay, generates consistent visual specifications/images, verifies the outputs, and exports the final production pack.

**Core flow:** Upload → Select exact culture/dialect → Extract → Normalize → Continuity checks → Human approval → Cultural plan → Screenplay rewrite → Visual generation → Verification → Export.

**Main benefit:** the system is not just a translation prompt. It uses structured memory, deterministic validation, human approval gates, traceability, culture-isolation, and selective asset retries to keep the adapted screenplay and visuals consistent.

This notebook is a complete **working MVP** for the *AI Engineering Hiring Challenge*. It supports one culture end-to-end, while keeping the design so another culture can run as an **independent project**.

---
## Pipeline (har step ke baad ek verification / approval gate)

```
 1 Upload ─► 2 Culture select ─► 3 Extract ─► 4 Normalize (canonical IDs) ─► 5 Continuity checks
                                                                              │
                         ┌───────────── GATE 1: user approves extraction ◄────┘
                         ▼
 6 Cultural adaptation PLAN ─► GATE 2: approve plan ─► GATE 3: approve costume plan
                         ▼
 7 Screenplay rewrite (traceable) ─► 8 Adaptation verification
                         ▼
 9 Visual specs ─► 10 Image generation (char → costume → scene, with references) ─► 11 Image verification
                         ▼
 12 Review dashboard ─► 13 Export pack ─► 14 Tests / limitations / AI-usage log
```

## Design principles (kyun yeh "agentic engineering" hai, sirf prompt nahi)
| Principle | Notebook mein kahan |
|---|---|
| Structured memory (canonical IDs) | Step 4 – `normalize()` raw LLM output ko `CHAR_/LOC_/PROP_/COST_` registry mein badalta hai |
| Deterministic checks (LLM pe bharosa nahi) | Step 5 – continuity validators, Step 8/11 – adaptation & image verifiers |
| Human approval gates (hash-based) | Step 6 – approval tab tak invalid ho jata hai jab upstream data badle |
| Retry ek asset ka, pura nahi | Step 10 – `retry_asset()` + spec-hash staleness |
| Traceability | Har adapted line/image → source scene + plan decision ID |
| Culture isolation | Har culture ka alag `Project` + alag workspace folder + leakage lint |
| Failure handling | `LLMFailure`, per-asset `failed/blocked` status, uncertainty register |

> **Mock mode:** agar `ANTHROPIC_API_KEY` set nahi hai, notebook bundled **sample data** se poora flow chalata hai (offline demo / tests). Live mode mein Claude se extraction, plan aur rewrite hota hai. Image backend: `openai` (gpt-image-1) ya `placeholder` (PIL cards jinke andar poori generation spec hoti hai).


---
# STEP 0 — Setup
### 0.1 Install & environment file


In [ ]:
# 0.1  (uncomment on first run)
# !pip install -q anthropic openai pydantic pandas ipywidgets pillow python-docx pypdf voila

from pathlib import Path
ENV_EXAMPLE = """# .env.example  -- copy to .env and fill in
ANTHROPIC_API_KEY=            # leave empty => MOCK mode with bundled sample data
ANTHROPIC_MODEL=claude-sonnet-5-5
OPENAI_API_KEY=               # needed only for real image generation
OPENAI_IMAGE_MODEL=gpt-image-1
IMAGE_BACKEND=placeholder     # placeholder | openai
STUDIO_WORKSPACE=workspace
"""
Path(".env.example").write_text(ENV_EXAMPLE)
print(ENV_EXAMPLE)

### 0.2 Imports, config, mode detection


In [ ]:
import os, re, json, hashlib, difflib, time, base64, io, zipfile, shutil, copy, html, textwrap
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

try:                                   # optional .env loader
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass

CFG = {
    "llm_model": os.getenv("ANTHROPIC_MODEL", "claude-sonnet-5-5"),
    "image_backend": os.getenv("IMAGE_BACKEND", "openai" if os.getenv("OPENAI_API_KEY") else "placeholder"),
    "openai_image_model": os.getenv("OPENAI_IMAGE_MODEL", "gpt-image-1"),
    "workspace": Path(os.getenv("STUDIO_WORKSPACE", "workspace")),
    "max_llm_retries": 2,
    "max_asset_attempts": 2,
}
MOCK_LLM = not os.getenv("ANTHROPIC_API_KEY")
CFG["workspace"].mkdir(parents=True, exist_ok=True)
print("LLM mode      :", "MOCK (bundled sample data)" if MOCK_LLM else f"LIVE ({CFG['llm_model']})")
print("Image backend :", CFG["image_backend"])
print("Workspace     :", CFG["workspace"].resolve())

### 0.3 Core utilities — hashing, AI-usage log, LLM wrapper with retries
`log_ai()` har LLM call ko `ai_usage_log.jsonl` mein likhta hai → yeh assignment ke **AI_USAGE.md** ka raw data hai.


In [ ]:
def slugify(s): return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")
def canon_json(o): return json.dumps(o, sort_keys=True, ensure_ascii=False, separators=(",", ":"), default=str)
def sha(o, n=12): return hashlib.sha256(canon_json(o).encode("utf-8")).hexdigest()[:n]
def utcnow(): return datetime.now(timezone.utc).isoformat(timespec="seconds")

USAGE_LOG = CFG["workspace"] / "ai_usage_log.jsonl"
def log_ai(purpose, **kw):
    with open(USAGE_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps({"ts": utcnow(), "purpose": purpose, **kw}, ensure_ascii=False) + "\n")

class LLMFailure(Exception):
    pass

def parse_json_loose(text):
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
    starts = [i for i in (text.find("{"), text.find("[")) if i >= 0]
    return json.loads(text[min(starts):] if starts else text)

def llm_json(purpose, system, user, mock=None, max_tokens=8000):
    """Call Claude and return parsed JSON. Retries with the parse error fed back.
    In MOCK mode returns mock() instead. Raises LLMFailure (caught by callers, never silent)."""
    if MOCK_LLM:
        if mock is None:
            raise LLMFailure(f"{purpose}: no mock available and no ANTHROPIC_API_KEY set")
        out = mock(); log_ai(purpose, mode="mock", ok=True); return out
    import anthropic
    client, prompt, last = anthropic.Anthropic(), user, None
    for attempt in range(CFG["max_llm_retries"] + 1):
        try:
            r = client.messages.create(model=CFG["llm_model"], max_tokens=max_tokens, system=system,
                                       messages=[{"role": "user", "content": prompt}])
            out = parse_json_loose("".join(b.text for b in r.content if b.type == "text"))
            log_ai(purpose, mode="live", model=CFG["llm_model"], ok=True, attempt=attempt,
                   in_tok=r.usage.input_tokens, out_tok=r.usage.output_tokens)
            return out
        except Exception as e:
            last = e
            log_ai(purpose, mode="live", ok=False, attempt=attempt, error=str(e)[:300])
            prompt = user + f"\n\nYour previous answer was invalid ({str(e)[:200]}). Return ONLY valid JSON."
    raise LLMFailure(f"{purpose} failed after retries: {last}")

---
# STEP 1 — Culture selection (exact dialect, not a broad label)

### 1.1 Selection model + validation
Assignment ka rule: *"Punjabi culture"*, *"Rajasthani look"*, *"North Indian village"* **too broad** hain. `validate_selection()` unko reject karta hai (jab tak user `allow_broad=True` na de).


In [ ]:
BROAD_LABELS = {"punjabi", "rajasthani", "north indian", "south indian", "indian", "hindi", "gujarati",
                "bengali", "tamil", "haryanvi", "bhojpuri", "marathi", "north indian village", "punjabi culture"}
SCRIPTS = {"Gurmukhi": (0x0A00, 0x0A7F), "Devanagari": (0x0900, 0x097F), "Bengali": (0x0980, 0x09FF),
           "Gujarati": (0x0A80, 0x0AFF), "Tamil": (0x0B80, 0x0BFF), "Latin": (0x0041, 0x024F)}

def validate_selection(sel, allow_broad=False):
    errs = []
    for k in ("culture", "dialect", "region", "setting", "script"):
        if not sel.get(k): errs.append(f"missing field: {k}")
    if sel.get("setting") not in ("rural", "urban"): errs.append("setting must be 'rural' or 'urban'")
    if sel.get("script") not in SCRIPTS: errs.append(f"script must be one of {list(SCRIPTS)}")
    if not allow_broad and slugify(sel.get("dialect", "")).replace("_", " ") in BROAD_LABELS:
        errs.append(f"'{sel.get('dialect')}' is too broad - give an exact dialect (e.g. Majhi, Malwai, Marwari, Mewari)")
    if errs: raise ValueError("; ".join(errs))
    return sel

SELECTION = {   # <-- change this for your assigned culture
    "culture": "Punjabi",
    "dialect": "Majhi",
    "region": "Majha (Amritsar-Gurdaspur-Tarn Taran belt), Punjab",
    "setting": "rural",
    "script": "Gurmukhi",
    "period": "contemporary (2020s), deep winter (Poh-Magh): wheat sown, mustard in flower",
}
validate_selection(SELECTION)
print(json.dumps(SELECTION, indent=2, ensure_ascii=False))

### 1.2 Cultural-mixing lint (isolation guard)
Ek **cheap deterministic guard**: har culture ke "marker words" hain; agar Majhi project ke text mein Marwari ya Haryanvi markers aaye → warning. Live mode mein isko ek LLM "cultural judge" ke saath pair kiya ja sakta hai (Step 8). Yeh *heuristic* hai — limitation `KNOWN_LIMITATIONS` mein likhi gayi hai.


In [ ]:
CULTURE_MARKERS = {
    "majhi":   ["chulha", "makki di roti", "sarson da saag", "manji", "phulkari", "pagri", "patka", "kada", "vehra", "bebe", "bapu ji", "gurdwara", "lassi"],
    "marwari": ["padharo", "ghoomar", "dhani", "bajra rotla", "safa", "angrakha", "thikana", "baisa", "bhanwar"],
    "mewari":  ["mewar", "udaipur", "chittor", "ghoomar", "hukam", "khamma ghani", "pichwai"],
    "malwai":  ["malwa", "dal bafla", "poha jalebi", "ujjain", "indori"],
    "bangru":  ["haryanvi", "ghagra", "hukka", "baithak", "ke re", "chaupal", "dhoti kurta"],
    "bhojpuri": ["litti", "chokha", "gamchha", "bhaiya ji"],
    "tamil":   ["veshti", "kolam", "filter coffee", "amma", "appa"],
}
def culture_key(sel): return slugify(sel["dialect"])

def mixing_lint(sel, text):
    """Return foreign-culture markers that appear in `text` for the selected culture."""
    me, low, hits = culture_key(sel), text.lower(), []
    own = set(CULTURE_MARKERS.get(me, []))
    for other, marks in CULTURE_MARKERS.items():
        if other == me: continue
        for m in marks:
            if m not in own and re.search(rf"\b{re.escape(m)}\b", low): hits.append((other, m))
    return hits

assert mixing_lint(SELECTION, "She wore a ghagra and offered bajra rotla") != []
assert mixing_lint(SELECTION, "She served makki di roti on a manji") == []
print("mixing lint ready")

---
# STEP 2 — Project container (independent per culture)
Each selected culture = **a separate `Project`** + a separate `workspace/<culture_slug>/` folder. There is no shared global state, so prompts/assets cannot leak across cultures.

State is stored in two layers (for reproducibility):
1. `raw` – LLM ka original extraction (kabhi mutate nahi hota, sirf `patch_scene` explicit patches se)
2. `alias_overrides`, `entity_edits` – user ke corrections (merge / rename / edit)
→ `normalize(raw + overrides)` hamesha **deterministically** canonical state banata hai.


In [ ]:
class Project:
    GATES = ("extraction", "plan", "costumes")

    def __init__(self, selection, source_text="", source_name="pasted", allow_broad=False):
        self.selection = validate_selection(dict(selection), allow_broad)
        self.slug = slugify(f"{selection['dialect']}_{selection['region'].split('(')[0]}_{selection['setting']}")
        self.dir = CFG["workspace"] / self.slug
        (self.dir / "images").mkdir(parents=True, exist_ok=True)
        self.source_text, self.source_name = source_text, source_name
        self.split = None           # deterministic scene split of the source
        self.raw = None             # LLM extraction (raw mentions)
        self.alias_overrides = {}   # nkey(mention) -> canonical name
        self.entity_edits = {}      # canonical id -> {field: value}
        self.edit_log = []          # every human/system edit (traceability)
        self.state = None           # normalized canonical state
        self.res_log = []           # how every mention was resolved
        self.issues = []            # continuity + extraction issues
        self.plan = None            # cultural adaptation plan
        self.adaptation = None      # adapted screenplay (per scene)
        self.approvals = {}         # gate -> {"hash":..., "by":..., "ts":...}
        self.assets = {}            # asset_id -> record
        self.failures = []

    def log(self, action, **kw):
        self.edit_log.append({"ts": utcnow(), "action": action, **kw})

    def save(self):
        d = {k: v for k, v in self.__dict__.items() if k != "dir"}
        (self.dir / "project.json").write_text(json.dumps(d, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

    @classmethod
    def load(cls, slug):
        d = json.loads((CFG["workspace"] / slug / "project.json").read_text(encoding="utf-8"))
        p = cls(d["selection"], d.get("source_text", ""), d.get("source_name", ""), allow_broad=True)
        p.__dict__.update({k: v for k, v in d.items() if k != "slug"})
        return p

print("Project class ready")

---
# STEP 3 — Upload: read the screenplay, preserve scene order, report extraction problems

### 3.1 Loader (paste / TXT / DOCX / PDF) + deterministic scene splitter
Scene splitting uses **regex** (not the LLM), so scene order is guaranteed. Problems (missing sluglines, scanned PDFs, scene count outside 3–5, duplicate sluglines) are **reported**, not silently ignored.


In [ ]:
SLUG_RE = re.compile(r"^\s*(INT\./EXT\.|INT/EXT\.|INT\.|EXT\.)\s*(.+?)\s*$", re.M | re.I)
CUE_COLON = re.compile(r"^\s*([A-Z][A-Z .'\-]{1,30}?)\s*(?:\(([^)]*)\))?\s*:\s*(\S.*)$", re.M)

def load_screenplay(src):
    """src: raw text, or a path to .txt/.docx/.pdf. Returns (text, problems)."""
    problems = []
    if isinstance(src, (str, Path)) and len(str(src)) < 400 and os.path.exists(str(src)):
        p = Path(src); ext = p.suffix.lower()
        if ext == ".txt": text = p.read_text(encoding="utf-8", errors="replace")
        elif ext == ".docx":
            import docx; text = "\n".join(par.text for par in docx.Document(str(p)).paragraphs)
        elif ext == ".pdf":
            from pypdf import PdfReader
            pages = [(pg.extract_text() or "") for pg in PdfReader(str(p)).pages]
            text = "\n".join(pages)
            if not text.strip(): problems.append({"severity": "error", "code": "PDF_NO_TEXT", "message": "PDF has no text layer (scanned?). OCR needed."})
        else: raise ValueError(f"unsupported file type {ext}")
    else:
        text = str(src)
    return text, problems

def dialogue_turns(text):
    return [(m.group(1).strip(), m.group(3).strip()) for m in CUE_COLON.finditer(text)]

def split_scenes(text):
    ms, scenes, problems = list(SLUG_RE.finditer(text)), [], []
    if not ms:
        problems.append({"severity": "error", "code": "NO_SLUGLINES", "message": "No INT./EXT. sluglines found - cannot split scenes."})
        return scenes, text, problems
    header = text[:ms[0].start()]
    seen = set()
    for i, m in enumerate(ms):
        body = text[m.end(): ms[i + 1].start() if i + 1 < len(ms) else len(text)].strip()
        parts = re.split(r"\s+[-\u2013\u2014]\s+", m.group(2).strip())
        loc, tm = (" - ".join(parts[:-1]), parts[-1]) if len(parts) > 1 else (m.group(2).strip(), "")
        if m.group(0).strip().upper() in seen:
            problems.append({"severity": "warning", "code": "DUP_SLUGLINE", "message": f"Duplicate slugline: {m.group(0).strip()}"})
        seen.add(m.group(0).strip().upper())
        if len(body.split()) < 8:
            problems.append({"severity": "warning", "code": "EMPTY_SCENE", "message": f"Scene {i+1} has almost no content"})
        scenes.append({"number": i + 1, "slugline": m.group(0).strip(), "int_ext": m.group(1).upper().rstrip("."),
                       "location": loc.title(), "time": tm.lower(), "text": body})
    if not 3 <= len(scenes) <= 5:
        problems.append({"severity": "warning", "code": "SCENE_COUNT", "message": f"{len(scenes)} scenes found; brief expects 3-5"})
    words = len(text.split())
    problems.append({"severity": "info", "code": "LENGTH", "message": f"{words} words (~{words/180:.1f} screenplay pages at 180 wpp)"})
    return scenes, header, problems

def ingest(project, src):
    text, p1 = load_screenplay(src)
    project.source_text = text
    project.split, header, p2 = split_scenes(text)
    project.header = header
    project.ingest_problems = p1 + p2
    project.log("ingest", scenes=len(project.split), problems=len(p1 + p2))
    return project.ingest_problems

### 3.2 Sample screenplay (neutral, 4 scenes, 4 characters)
It contains **three deliberate continuity traps** so detection can be demonstrated in Step 5:
* **Alias trap:** `AMAR SINGH` / `AMAR` / `THE ELDER SON` / `A. SINGH` → ek hi character
* **Prop trap:** letter Scene 2 mein *Karan* ko di jati hai, par Scene 3 mein Amar ke paas *magically wapas* hai
* **Costume trap:** Amar ki shirt Scene 3 mein bina reason badal jati hai

To use your own 3–5 page screenplay, enter a file path or text in `SCREENPLAY_SOURCE`.


In [ ]:
SAMPLE_SCREENPLAY = """THE LETTER

CAST
AMAR SINGH (28) - the elder son, back from the city
MOTHER (55) - Amar's mother
FATHER (60) - a quiet farmer
KARAN (19) - Amar's younger brother

INT. FAMILY KITCHEN - NIGHT

A single bulb hums. MOTHER stirs a pot on the stove. FATHER stands in the doorway, wiping his hands. AMAR enters with a duffel bag, wearing a grey city shirt and jeans. A folded LETTER addressed to A. SINGH sticks out of his pocket.

MOTHER: You've become thin. Eat.
AMAR: I ate on the way.

She serves him anyway. FATHER watches, says nothing, and leaves.

MOTHER (quietly): He waited up for you all night.

Amar tucks the letter deeper into his pocket.

EXT. FIELD PATH - MORNING

Mist hangs over the crops. AMAR and his younger brother KARAN walk along the edge of a field. Karan notices the letter.

KARAN: What is that paper?
AMAR: Nothing. Keep it safe for me. And don't tell Father.

Amar hands the LETTER to Karan, who pockets it. A tractor rumbles past; they watch it go.

KARAN: Are you going back to the city?
AMAR: I don't know.

EXT. COURTYARD - EVENING

FATHER sits on a cot with a cup of tea. THE ELDER SON stands before him in a clean white shirt and trousers.

AMAR: I lost my job. Three months ago.

A long silence. Father puts the cup down and stares at the horizon. Amar takes the LETTER from his pocket and offers it. Father reads it slowly, folds it, keeps it, and pats the cot beside him.

FATHER: Sit.

Amar sits.

FATHER: Lies make the load heavier. Next time, carry it with us.

EXT. VILLAGE BUS STOP - DAY

Next morning. A bus idles. AMAR, still in the white shirt, holds his duffel bag. KARAN watches from his bicycle. FATHER, the letter still in his pocket, unstraps his old WRISTWATCH and fastens it on Amar's wrist.

FATHER: The farm will wait. Be here for the harvest.

Amar looks at the bus, then sets the bag down.

AMAR: I'm not going anywhere, Father.

Karan grins. Father nods once.
"""

SCREENPLAY_SOURCE = SAMPLE_SCREENPLAY      # <-- or a path: "my_script.pdf" / "my_script.docx" / "my_script.txt"

---
# STEP 4 — Structured extraction (scene-by-scene records)

### 4.1 Raw extraction schema + prompts
Extraction **do passes** mein hota hai: (a) poora cast + props catalog (aliases ke saath), (b) **ek scene per call** — taaki ek scene fail ho to sirf wahi retry ho (`reextract_scene`).


In [ ]:
SCENE_SCHEMA = """{
 "number": int, "int_ext": "INT|EXT|INT/EXT", "location": str, "sub_location": str, "time": str, "day": str,
 "weather": str, "mood": str, "summary": str, "dramatic_purpose": str,
 "characters": [mention, ...],                       // everyone physically present
 "costumes": {mention: "garments, footwear, headwear, jewellery as stated or clearly implied"},
 "costume_change_reason": {mention: str},            // ONLY if the source gives a reason for a change
 "props_in": [{"prop": str, "holder": mention|"NONE"}],   // props visible at scene START and who holds them
 "transfers": [{"prop": str, "from": mention, "to": mention|"GONE"|"CONSUMED"}],   // in order
 "knowledge_gain": [{"char": mention, "fact": str}], "knowledge_use": [{"char": mention, "fact": str}],
 "injuries": [{"char": mention, "injury": str, "status": "new|ongoing|healed"}],
 "production": {"set": str, "food": [..], "vehicles": [..], "animals": [..], "extras": [..], "rituals": [..], "gestures": [..], "sound": [..]},
 "emotional_change": {mention: "from X to Y"}
}"""

CAST_SCHEMA = """{"characters": [{"name": str, "aliases": [str], "age": int|null, "role": str, "relationships": {name: relation},
                  "personality": str, "dialect": "", "emotional_state": str}],
 "props": [{"name": str, "aliases": [str], "story_critical": bool}]}"""

EXTRACT_SYSTEM = ("You are a meticulous script supervisor building a continuity bible. Output ONLY valid JSON. "
  "Never invent facts: use '' or [] when the source is silent. Merge every alias/nickname/role-label of one person "
  "into ONE character. If a costume is not restated in a scene, repeat the previous scene's costume (unless the source "
  "states a change). Track exactly who holds each prop at the START of each scene and every hand-over.")

def prompt_cast(project):
    return f"{CAST_SCHEMA}\n\nSCREENPLAY HEADER + FULL TEXT:\n{project.source_text}"

def prompt_scene(project, sc, cast):
    prev = [s for s in project.split if s["number"] == sc["number"] - 1]
    return (f"Known cast (canonical names + aliases): {json.dumps(cast, ensure_ascii=False)}\n"
            f"Previous scene text (for continuity): {prev[0]['text'] if prev else 'NONE'}\n\n"
            f"Extract scene {sc['number']} as JSON with this schema:\n{SCENE_SCHEMA}\n\n"
            f"SLUGLINE: {sc['slugline']}\nSCENE TEXT:\n{sc['text']}")

### 4.2 Bundled mock extraction (sample screenplay) + heuristic fallback (any other screenplay, offline)
Mock data **LLM output ki tarah raw mentions** use karta hai (`"Amar"`, `"The Elder Son"`, `"A. Singh"`), taaki normalization ka kaam asli ho.


In [ ]:
def _scene(number, int_ext, location, time, day, weather, mood, summary, purpose, characters, costumes,
           props_in, transfers=(), gain=(), use=(), reasons=None, production=None, emo=None, sub=""):
    return {"number": number, "int_ext": int_ext, "location": location, "sub_location": sub, "time": time, "day": day,
            "weather": weather, "mood": mood, "summary": summary, "dramatic_purpose": purpose,
            "characters": list(characters), "costumes": costumes, "costume_change_reason": reasons or {},
            "props_in": [{"prop": p, "holder": h} for p, h in props_in],
            "transfers": [{"prop": p, "from": a, "to": b} for p, a, b in transfers],
            "knowledge_gain": [{"char": c, "fact": f} for c, f in gain], "knowledge_use": [{"char": c, "fact": f} for c, f in use],
            "injuries": [], "production": production or {}, "emotional_change": emo or {}}

SAMPLE_RAW = {
 "characters": [
  {"name": "Amar Singh", "aliases": ["Amar", "the elder son", "A. Singh"], "age": 28, "role": "Elder son, just back from the city",
   "relationships": {"Mother": "son", "Father": "son", "Karan": "elder brother"},
   "personality": "proud, avoidant, ashamed but loving", "dialect": "", "emotional_state": "avoidance -> admission -> belonging"},
  {"name": "Mother", "aliases": ["Amar's mother"], "age": 55, "role": "Mother; shows love through food",
   "relationships": {"Amar Singh": "mother", "Father": "wife", "Karan": "mother"},
   "personality": "warm, observant, indirect", "dialect": "", "emotional_state": "worried, tender"},
  {"name": "Father", "aliases": ["the farmer"], "age": 60, "role": "Quiet farmer, head of household",
   "relationships": {"Amar Singh": "father", "Karan": "father", "Mother": "husband"},
   "personality": "restrained, proud, deeply caring", "dialect": "", "emotional_state": "silent hurt -> acceptance"},
  {"name": "Karan", "aliases": ["younger brother"], "age": 19, "role": "Younger brother",
   "relationships": {"Amar Singh": "younger brother"},
   "personality": "curious, loyal, light-hearted", "dialect": "", "emotional_state": "curious -> relieved"}],
 "props": [
  {"name": "Letter", "aliases": ["folded letter"], "story_critical": True},
  {"name": "Duffel bag", "aliases": ["bag"], "story_critical": False},
  {"name": "Wristwatch", "aliases": ["old watch", "watch"], "story_critical": True}],
 "scenes": [
  _scene(1, "INT", "Family Kitchen", "night", "Day 1", "", "tense, tender",
         "Amar returns; Mother feeds him; Father watches silently; Amar hides a letter.",
         "Establish Amar's secret and the family's unspoken love.",
         ["Amar Singh", "Mother", "Father"],
         {"Amar Singh": "grey city shirt, jeans, sneakers", "Mother": "plain salwar kameez, dupatta over head", "Father": "kurta pyjama, turban, sweater"},
         [("duffel bag", "Amar Singh"), ("folded letter", "Amar Singh")],
         production={"food": ["home-cooked dinner"], "gestures": ["Mother serves anyway", "Father silent in doorway"], "sound": ["bulb hum"]},
         emo={"Amar Singh": "from weary to guarded"}),
  _scene(2, "EXT", "Field Path", "morning", "Day 2", "mist", "uneasy, intimate",
         "Amar asks Karan to hold the letter secret from Father.",
         "Raise the stakes: the secret spreads to a second person.",
         ["Amar", "Karan"],
         {"Amar": "grey shirt, blue jeans, sneakers", "Karan": "checked shirt, jeans, woollen cap"},
         [("letter", "Amar")], transfers=[("letter", "Amar", "Karan")],
         gain=[("Karan", "Amar is hiding a letter")],
         production={"vehicles": ["tractor"], "sound": ["tractor rumble"]},
         emo={"Amar": "from guarded to anxious"}),
  _scene(3, "EXT", "Courtyard", "evening", "Day 2", "", "heavy, then releasing",
         "Amar admits the job loss; Father accepts silently and invites him to sit.",
         "The admission and the first act of acceptance (emotional turn).",
         ["The Elder Son", "Father"],
         {"The Elder Son": "clean white shirt, trousers", "Father": "kurta pyjama, turban, sweater"},
         [("letter", "The Elder Son")], transfers=[("letter", "The Elder Son", "Father")],
         gain=[("Father", "Amar lost his job")],
         production={"set": "cot in courtyard", "food": ["tea"], "gestures": ["pats cot beside him"]},
         emo={"The Elder Son": "from anxious to admitting"}),
  _scene(4, "EXT", "Village Bus Stop", "day", "Day 3", "", "quiet resolution",
         "Father gives Amar his watch; Amar chooses to stay.",
         "Resolution: belonging replaces avoidance.",
         ["A. Singh", "Father", "Karan"],
         {"A. Singh": "clean white shirt, trousers", "Father": "kurta pyjama, turban, sweater, shawl", "Karan": "checked shirt, jeans, cap"},
         [("letter", "Father"), ("old watch", "Father"), ("duffel bag", "A. Singh")],
         transfers=[("old watch", "Father", "A. Singh")],
         use=[("Father", "Amar lost his job")],
         production={"vehicles": ["bus", "bicycle"], "gestures": ["Father fastens watch on wrist"]},
         emo={"A. Singh": "from admitting to belonging"})]}

CAST_LINE = re.compile(r"^\s*([A-Z][A-Z .']+?)\s*\((\d+)\)\s*[-\u2013]\s*(.+)$", re.M)

def heuristic_cast(project):
    chars = [{"name": n.title(), "aliases": [], "age": int(a), "role": r.strip(), "relationships": {}, "personality": "",
              "dialect": "", "emotional_state": ""} for n, a, r in CAST_LINE.findall(project.source_text)]
    known = {c["name"].upper() for c in chars}
    for sc in project.split:
        for n, _ in dialogue_turns(sc["text"]):
            if n not in known: known.add(n); chars.append({"name": n.title(), "aliases": [], "age": None, "role": "", "relationships": {},
                                                           "personality": "", "dialect": "", "emotional_state": ""})
    return {"characters": chars, "props": []}

def heuristic_scene(project, sc):
    names = []
    for n, _ in dialogue_turns(sc["text"]):
        if n.title() not in names: names.append(n.title())
    first = re.split(r"(?<=[.!?])\s", sc["text"].strip())[0] if sc["text"].strip() else ""
    return _scene(sc["number"], sc["int_ext"], sc["location"], sc["time"], "", "", "", first, "", names, {}, [])

def is_sample(project): return sha(project.source_text) == sha(SAMPLE_SCREENPLAY)

### 4.3 Run extraction (per-scene calls, failures isolated, single-scene retry)


In [ ]:
def run_extraction(project, only_scenes=None):
    cast = llm_json("extract_cast", EXTRACT_SYSTEM, prompt_cast(project),
                    mock=lambda: (copy.deepcopy({"characters": SAMPLE_RAW["characters"], "props": SAMPLE_RAW["props"]}) if is_sample(project) else heuristic_cast(project)))
    cast_hint = [{"name": c["name"], "aliases": c.get("aliases", [])} for c in cast["characters"]]
    scenes = {s["number"]: s for s in (project.raw or {}).get("scenes", [])} if only_scenes else {}
    for sc in project.split:
        if only_scenes and sc["number"] not in only_scenes: continue
        try:
            rec = llm_json(f"extract_scene_{sc['number']}", EXTRACT_SYSTEM, prompt_scene(project, sc, cast_hint),
                           mock=lambda sc=sc: (copy.deepcopy(SAMPLE_RAW["scenes"][sc["number"] - 1]) if is_sample(project) else heuristic_scene(project, sc)))
            rec["number"] = sc["number"]; scenes[sc["number"]] = rec
        except LLMFailure as e:
            project.failures.append({"stage": "extraction", "scene": sc["number"], "error": str(e)})
            scenes[sc["number"]] = {"number": sc["number"], "_failed": True, "location": sc["location"], "time": sc["time"], "int_ext": sc["int_ext"]}
    project.raw = {"characters": cast["characters"], "props": cast.get("props", []), "scenes": [scenes[k] for k in sorted(scenes)]}
    project.log("extraction", only=only_scenes or "all")

def reextract_scene(project, number):
    """Retry ONE scene without touching the rest, then re-normalize."""
    keep = {k: v for k, v in project.raw.items()}
    run_extraction(project, only_scenes=[number])
    normalize(project)
    return project.raw["scenes"][number - 1]

---
# STEP 5 — Normalize into canonical records (no duplicate entities)

### 5.1 Entity resolver: aliases → ONE canonical ID
Resolution order: **exact alias** → **token subset** (`Amar` ⊂ `Amar Singh`) → **fuzzy ≥ 0.88** → create new (aur *implicit entity* warning). Ambiguous match (2+ candidates) kabhi guess nahi hota — flag hota hai. Har decision `res_log` mein jata hai (traceability).


In [ ]:
STOP = {"the", "an", "of", "mr", "mrs", "ms"}
def norm_tokens(s):
    s = re.sub(r"\b\w+'s\b", " ", str(s).lower())
    return [t for t in re.sub(r"[^a-z0-9 ]", " ", s).split() if t not in STOP]
def nkey(s): return " ".join(norm_tokens(s))

class Resolver:
    def __init__(self, prefix, log, overrides=None):
        self.prefix, self.log, self.ov = prefix, log, overrides or {}
        self.ents, self.index, self.ambiguous = {}, {}, []

    def _new_id(self, name):
        toks = [t.upper() for t in norm_tokens(name)] or ["X"]
        base = f"{self.prefix}_{toks[0]}" if self.prefix == "CHAR" else f"{self.prefix}_{'_'.join(toks[:3])}"
        cid, i = base, 1
        while cid in self.ents: i += 1; cid = f"{base}_{i}"
        return cid

    def add_alias(self, cid, a):
        k = nkey(a)
        if not k: return
        if self.index.get(k, cid) != cid:
            self.ambiguous.append({"mention": a, "candidates": sorted({cid, self.index[k]}), "ctx": "alias-collision"}); return
        self.index[k] = cid
        e = self.ents[cid]
        if a.strip().lower() != e["name"].lower() and a not in e["aliases"]: e["aliases"].append(a)

    def _create(self, name):
        cid = self._new_id(name); self.ents[cid] = {"name": name, "aliases": []}; self.add_alias(cid, name); return cid

    def find(self, mention, create=False, ctx=""):
        mention = self.ov.get(nkey(mention), mention)
        k = nkey(mention)
        if not k: return None
        if k in self.index: return self.index[k]
        toks = set(k.split())
        cands = {cid for key, cid in self.index.items() if toks <= set(key.split()) or set(key.split()) <= toks}
        if len(cands) == 1:
            cid = next(iter(cands)); self.add_alias(cid, mention)
            self.log.append({"kind": self.prefix, "mention": mention, "id": cid, "how": "token-subset", "ctx": ctx}); return cid
        if len(cands) > 1:
            self.ambiguous.append({"mention": mention, "candidates": sorted(cands), "ctx": ctx}); return None
        best = max(((difflib.SequenceMatcher(None, k, key).ratio(), cid) for key, cid in self.index.items()), default=(0, None))
        if best[0] >= 0.88:
            self.add_alias(best[1], mention)
            self.log.append({"kind": self.prefix, "mention": mention, "id": best[1], "how": f"fuzzy={best[0]:.2f}", "ctx": ctx}); return best[1]
        if create:
            cid = self._create(mention)
            self.log.append({"kind": self.prefix, "mention": mention, "id": cid, "how": "implicit", "ctx": ctx}); return cid
        return None

    def declare(self, name, aliases=()):
        cid = self.find(name, create=False, ctx="declare") or self._create(name)
        self.add_alias(cid, name)
        for a in aliases: self.add_alias(cid, a)
        return cid

COST_STOP = {"a", "an", "the", "and", "with", "over", "in", "of", "his", "her"}
def cost_tokens(d): return {t for t in re.findall(r"[a-z]+", str(d).lower()) if t not in COST_STOP}

class CostumeBook:
    """One canonical costume per (character, garment-set). Jaccard >= 0.6 => same costume."""
    def __init__(self): self.items = {}
    def get(self, char_id, desc):
        t, best = cost_tokens(desc), None
        for c in self.items.values():
            if c["char_id"] != char_id: continue
            j = len(t & c["tokens"]) / max(1, len(t | c["tokens"]))
            if j >= 0.6 and (best is None or j > best[0]): best = (j, c["id"])
        if best:
            c = self.items[best[1]]
            if len(desc) > len(c["description"]): c["description"] = desc
            return c["id"]
        n = 1 + sum(1 for c in self.items.values() if c["char_id"] == char_id)
        cid = f"COST_{char_id[5:]}_{n:02d}"
        self.items[cid] = {"id": cid, "char_id": char_id, "description": desc, "tokens": t}
        return cid

### 5.2 `normalize(project)` — raw LLM output + user overrides → canonical state


In [ ]:
HOLDER_TOKENS = {"GONE", "CONSUMED", "NONE", ""}

def normalize(project):
    raw, ov, log = copy.deepcopy(project.raw), project.alias_overrides, []
    RC, RL, RP, CB = Resolver("CHAR", log, ov), Resolver("LOC", log, ov), Resolver("PROP", log, ov), CostumeBook()
    meta, pmeta = {}, {}
    decl = raw.get("characters", [])
    ordered = [c for c in decl if nkey(c["name"]) not in ov] + [c for c in decl if nkey(c["name"]) in ov]
    for c in ordered:                                     # declared characters; overridden ones become aliases
        cid = RC.declare(c["name"], c.get("aliases", []))
        m = meta.setdefault(cid, {})
        for k, v in c.items():
            if k not in ("name", "aliases") and v not in (None, "", [], {}) and k not in m: m[k] = v
    for p in raw.get("props", []):
        pid = RP.declare(p["name"], p.get("aliases", []))
        pmeta.setdefault(pid, {"story_critical": False})["story_critical"] |= bool(p.get("story_critical"))

    def C(m, ctx):
        return RC.find(m, create=True, ctx=ctx)
    def H(m, ctx):
        return m if m in HOLDER_TOKENS or m is None else C(m, ctx)

    scenes = []
    for i, s in enumerate(raw.get("scenes", []), 1):
        n = s.get("number", i); sid = f"SC{n:02d}"
        if s.get("_failed"):
            scenes.append({"scene_id": sid, "number": n, "_failed": True, "location_id": None, "characters": [], "costume_ids": {}, "costume_change_reason": {},
                           "props_in": [], "transfers": [], "knowledge_gain": [], "knowledge_use": [], "injuries": [], "emotional_change": {},
                           "int_ext": s.get("int_ext", ""), "location": s.get("location", ""), "time": s.get("time", "")}); continue
        chars = []
        for m in s.get("characters", []):
            cid = C(m, sid)
            if cid and cid not in chars: chars.append(cid)
        costume_ids = {}
        for m, d in s.get("costumes", {}).items():
            cid = C(m, sid)
            if cid: costume_ids[cid] = CB.get(cid, d)
        scenes.append({
            "scene_id": sid, "number": n, "int_ext": s.get("int_ext", ""), "location": s.get("location", ""),
            "location_id": RL.find(s.get("location", "UNKNOWN"), create=True, ctx=sid), "sub_location": s.get("sub_location", ""),
            "time": s.get("time", ""), "day": s.get("day", ""), "weather": s.get("weather", ""), "mood": s.get("mood", ""),
            "summary": s.get("summary", ""), "dramatic_purpose": s.get("dramatic_purpose", ""),
            "characters": chars, "costume_ids": costume_ids,
            "costume_change_reason": {C(m, sid): r for m, r in s.get("costume_change_reason", {}).items()},
            "props_in": [{"prop_id": RP.find(p["prop"], True, sid), "holder_id": H(p.get("holder"), sid)} for p in s.get("props_in", [])],
            "transfers": [{"prop_id": RP.find(t["prop"], True, sid), "from": H(t.get("from"), sid), "to": H(t.get("to"), sid)} for t in s.get("transfers", [])],
            "knowledge_gain": [{"char_id": C(k["char"], sid), "fact": k["fact"]} for k in s.get("knowledge_gain", [])],
            "knowledge_use": [{"char_id": C(k["char"], sid), "fact": k["fact"]} for k in s.get("knowledge_use", [])],
            "injuries": [{"char_id": C(k["char"], sid), "injury": k["injury"], "status": k.get("status", "ongoing")} for k in s.get("injuries", [])],
            "emotional_change": {C(m, sid): t for m, t in s.get("emotional_change", {}).items()},
            "production": s.get("production", {}),
            "source_text": next((x["text"] for x in project.split if x["number"] == n), "")})

    chars = {}
    for cid, e in RC.ents.items():
        sc_ids = [s["scene_id"] for s in scenes if cid in s["characters"]]
        rel = {(RC.find(k, create=False) or k): v for k, v in meta.get(cid, {}).get("relationships", {}).items()}
        rec = {"id": cid, "name": e["name"], "aliases": e["aliases"], **{k: v for k, v in meta.get(cid, {}).items() if k != "relationships"},
               "relationships": rel, "scenes": sc_ids, "entrance": sc_ids[0] if sc_ids else "", "exit": sc_ids[-1] if sc_ids else ""}
        chars[cid] = rec
    locs = {lid: {"id": lid, "name": e["name"], "aliases": e["aliases"],
                  "sub_locations": sorted({s["sub_location"] for s in scenes if s["location_id"] == lid and s["sub_location"]}),
                  "int_ext": sorted({s["int_ext"] for s in scenes if s["location_id"] == lid}),
                  "scenes": [s["scene_id"] for s in scenes if s["location_id"] == lid]} for lid, e in RL.ents.items()}
    props = {pid: {"id": pid, "name": e["name"], "aliases": e["aliases"], "story_critical": pmeta.get(pid, {}).get("story_critical", False),
                   "scenes": [s["scene_id"] for s in scenes if any(p["prop_id"] == pid for p in s["props_in"] + s["transfers"])]}
             for pid, e in RP.ents.items()}
    costs = {cid: {"id": cid, "char_id": c["char_id"], "description": c["description"],
                   "scenes": [s["scene_id"] for s in scenes if s["costume_ids"].get(c["char_id"]) == cid]} for cid, c in CB.items.items()}
    for eid, edits in project.entity_edits.items():          # User edits (rename, age, role ...)
        for reg in (chars, locs, props, costs):
            if eid in reg: reg[eid].update(edits)
    norm_issues = [{"code": "IMPLICIT_ENTITY", "severity": "warning", "scenes": [l["ctx"]], "entities": [l["id"]], "fix_hint": "",
                    "message": f"'{l['mention']}' was not declared in the cast/catalog; created {l['id']} implicitly - verify it is not a duplicate."}
                   for l in log if l["how"] == "implicit"]
    for R in (RC, RL, RP):
        for a in R.ambiguous:
            norm_issues.append({"code": "AMBIGUOUS_ALIAS", "severity": "error", "scenes": [a["ctx"]], "entities": a["candidates"], "fix_hint": "merge_entities(P, keep_id, drop_id) or rename",
                                "message": f"'{a['mention']}' could refer to {a['candidates']} - merge or rename to resolve."})
    project.state = {"chars": chars, "locs": locs, "props": props, "costs": costs, "scenes": scenes, "norm_issues": norm_issues}
    project.res_log = log
    return project.state

### 5.3 Driver: create the project, ingest the sample, extract, normalize


In [ ]:
P = Project(SELECTION)
problems = ingest(P, SCREENPLAY_SOURCE)
print("Scenes found:", [(s["number"], s["slugline"]) for s in P.split])
for pr in problems: print(f"  [{pr['severity']}] {pr['code']}: {pr['message']}")

run_extraction(P)
normalize(P)
print("\nCanonical characters:", {k: v["name"] for k, v in P.state["chars"].items()})
print("Aliases merged      :", {k: v["aliases"] for k, v in P.state["chars"].items()})
print("Locations           :", list(P.state["locs"]))
print("Props               :", list(P.state["props"]))
print("Costumes            :", {k: v["description"] for k, v in P.state["costs"].items()})

### 5.4 Review tables (yeh wahi data hai jo dashboard ka "Scene extraction" screen dikhata hai)


In [ ]:
def _names(st, ids): return ", ".join(st["chars"][i]["name"] if i in st["chars"] else str(i) for i in ids)

def scene_table(st):
    rows = []
    for s in st["scenes"]:
        rows.append({"scene": s["scene_id"], "INT/EXT": s.get("int_ext", ""), "location_id": s.get("location_id"), "time": s.get("time", ""),
                     "characters": ", ".join(s["characters"]), "costumes": ", ".join(f"{c}->{k}" for c, k in s["costume_ids"].items()),
                     "props_in": ", ".join(f"{p['prop_id']}@{p['holder_id']}" for p in s["props_in"]),
                     "transfers": ", ".join(f"{t['prop_id']}:{t['from']}->{t['to']}" for t in s["transfers"]),
                     "summary": s.get("summary", ""), "purpose": s.get("dramatic_purpose", "")})
    return pd.DataFrame(rows)

def char_table(st):
    return pd.DataFrame([{"id": c["id"], "name": c["name"], "aliases": "; ".join(c["aliases"]), "age": c.get("age"), "role": c.get("role", ""),
                          "entrance": c["entrance"], "exit": c["exit"], "scenes": ",".join(c["scenes"])} for c in st["chars"].values()])

def costume_table(st):
    return pd.DataFrame([{"id": c["id"], "character": c["char_id"], "description": c["description"], "scenes": ",".join(c["scenes"])} for c in st["costs"].values()])

pd.set_option("display.max_colwidth", 70, "display.width", 200)
display(scene_table(P.state)); display(char_table(P.state)); display(costume_table(P.state))

---
# STEP 6 — Verification tools: duplicate, extraction & continuity checks
These are **deterministic validators**, not the LLM. Every issue includes: `code`, `severity`, **affected scenes**, entities aur `fix_hint`.

### 6.1 Extraction integrity + duplicate detection


In [ ]:
def issue(code, sev, msg, scenes=(), entities=(), hint=""):
    return {"code": code, "severity": sev, "message": msg, "scenes": list(dict.fromkeys(scenes)), "entities": list(entities), "fix_hint": hint}

def char_index(st):
    idx = {}
    for c in st["chars"].values():
        for n in [c["name"]] + c["aliases"]: idx[nkey(n)] = c["id"]
    return idx

def resolve_speaker(idx, name):
    k = nkey(name)
    if k in idx: return idx[k]
    toks = set(k.split())
    c = {cid for key, cid in idx.items() if toks and (toks <= set(key.split()) or set(key.split()) <= toks)}
    return next(iter(c)) if len(c) == 1 else None

def check_extraction_integrity(project):
    st, iss = project.state, []
    for p in getattr(project, "ingest_problems", []):
        if p["severity"] in ("error", "warning"): iss.append(issue("INGEST_" + p["code"], p["severity"], p["message"]))
    if len(st["scenes"]) != len(project.split):
        iss.append(issue("MISSING_SCENE", "error", f"Source has {len(project.split)} scenes but extraction produced {len(st['scenes'])}"))
    nums = [s["number"] for s in st["scenes"]]
    if nums != list(range(1, len(nums) + 1)):
        iss.append(issue("SCENE_ORDER", "error", f"Scene numbers not contiguous/ordered: {nums}"))
    for s in st["scenes"]:
        if s.get("_failed"):
            iss.append(issue("SCENE_EXTRACTION_FAILED", "error", f"{s['scene_id']} extraction failed - retry with reextract_scene()", [s["scene_id"]])); continue
        if not s["characters"]: iss.append(issue("NO_CHARACTERS", "warning", f"{s['scene_id']} lists no characters", [s["scene_id"]]))
    for c in st["chars"].values():
        if not c["scenes"]: iss.append(issue("UNUSED_CHARACTER", "warning", f"{c['id']} ({c['name']}) never appears in a scene", entities=[c["id"]]))
    idx = char_index(st)                                  # speakers in the SOURCE text must be canonical + present
    for s in st["scenes"]:
        for spk, _ in dialogue_turns(s.get("source_text", "")):
            cid = resolve_speaker(idx, spk)
            if cid is None: iss.append(issue("UNKNOWN_SPEAKER", "error", f"Speaker '{spk}' in {s['scene_id']} matches no canonical character", [s["scene_id"]]))
            elif cid not in s["characters"]: iss.append(issue("SPEAKER_NOT_IN_SCENE", "error", f"{cid} speaks in {s['scene_id']} but is not listed as present", [s["scene_id"]], [cid]))
    return iss

def check_duplicates(st):
    iss = []
    for kind, reg in (("character", st["chars"]), ("prop", st["props"]), ("location", st["locs"])):
        items = list(reg.values())
        for i in range(len(items)):
            for j in range(i + 1, len(items)):
                a, b = items[i], items[j]
                r = difflib.SequenceMatcher(None, nkey(a["name"]), nkey(b["name"])).ratio()
                if r >= 0.85: iss.append(issue("DUPLICATE_" + kind.upper(), "error", f"{a['id']} and {b['id']} look like the same {kind} (similarity {r:.2f})",
                                               entities=[a["id"], b["id"]], hint=f"merge_entities(P, '{a['id']}', '{b['id']}')"))
    cs = list(st["costs"].values())
    for i in range(len(cs)):
        for j in range(i + 1, len(cs)):
            if cs[i]["char_id"] == cs[j]["char_id"]:
                ta, tb = cost_tokens(cs[i]["description"]), cost_tokens(cs[j]["description"])
                if len(ta & tb) / max(1, len(ta | tb)) >= 0.6:
                    iss.append(issue("DUPLICATE_COSTUME", "error", f"{cs[i]['id']} and {cs[j]['id']} are near-identical costumes", entities=[cs[i]["id"], cs[j]["id"]]))
    return iss

### 6.2 State-tracking continuity checks (props · costume · knowledge · injuries)
Core idea: scenes ke upar ek **state machine** chalti hai — `holder[prop]`, `last_costume[char]`, `knows`, `injuries` — aur har scene ke start/end state ko compare karti hai.


In [ ]:
NOBODY = (None, "", "NONE")
GONE = ("GONE", "CONSUMED")

def check_continuity(st):
    iss, holder, hist = [], {}, {}
    cost_last, know, inj = {}, set(), {}
    crit = {pid: p["story_critical"] for pid, p in st["props"].items()}
    for sc in st["scenes"]:
        if sc.get("_failed"): continue
        sid, present, mentioned = sc["scene_id"], set(sc["characters"]), set()
        # ---- props at scene start
        for p in sc["props_in"]:
            pid, h = p["prop_id"], p["holder_id"]; mentioned.add(pid)
            if pid in holder:
                exp = holder[pid]
                if exp in GONE:
                    iss.append(issue("PROP_AFTER_GONE", "error", f"{pid} was {exp.lower()} earlier but reappears in {sid}", hist[pid][-1:] + [sid], [pid]))
                elif exp != h and not (exp in NOBODY and h in NOBODY):
                    iss.append(issue("PROP_TELEPORT", "error",
                        f"{pid} ended {hist[pid][-1]} with {exp} but starts {sid} with {h} - no hand-over explains it",
                        hist[pid][-1:] + [sid], [pid, str(exp), str(h)],
                        "Add the missing transfer in the earlier scene (patch_scene) or correct the holder"))
            if h not in NOBODY and h not in GONE and h not in present:
                iss.append(issue("PROP_HOLDER_ABSENT", "error", f"{pid} is held by {h} who is not in {sid}", [sid], [pid, h]))
            holder[pid] = h; hist.setdefault(pid, []).append(sid)
        # ---- transfers (ordered)
        for t in sc["transfers"]:
            pid = t["prop_id"]
            if holder.get(pid) != t["from"]:
                iss.append(issue("PROP_TRANSFER_INVALID", "error", f"{sid}: {pid} handed over by {t['from']} but current holder is {holder.get(pid)}", [sid], [pid]))
            for who in (t["from"], t["to"]):
                if who not in NOBODY and who not in GONE and who not in present:
                    iss.append(issue("TRANSFER_PARTY_ABSENT", "error", f"{sid}: {who} takes part in a hand-over of {pid} but is not in the scene", [sid], [pid, who]))
            holder[pid] = t["to"]; hist.setdefault(pid, []).append(sid)
        for pid, h in holder.items():                    # a story-critical prop silently vanishing
            if crit.get(pid) and h in present and pid not in mentioned and not any(t["prop_id"] == pid for t in sc["transfers"]):
                iss.append(issue("PROP_VANISHED", "warning", f"{pid} is held by {h} (present in {sid}) but not mentioned", [sid], [pid]))
        # ---- costume continuity
        for cid in sc["characters"]:
            cost = sc["costume_ids"].get(cid)
            if not cost:
                iss.append(issue("COSTUME_MISSING", "warning", f"{cid} has no costume in {sid}", [sid], [cid])); continue
            last = cost_last.get(cid)
            if last and last[0] != cost and cid not in sc["costume_change_reason"]:
                same_day = last[2] == sc.get("day")
                iss.append(issue("COSTUME_CHANGE_UNEXPLAINED", "error" if same_day else "warning",
                    f"{cid} changes {last[0]} -> {cost} between {last[1]} and {sid} {'on the same day ' if same_day else ''}without a story reason",
                    [last[1], sid], [cid, last[0], cost], "Add costume_change_reason in the later scene, or fix the costume description"))
            cost_last[cid] = (cost, sid, sc.get("day"))
        # ---- knowledge
        for k in sc["knowledge_gain"]: know.add((k["char_id"], nkey(k["fact"])))
        for k in sc["knowledge_use"]:
            if (k["char_id"], nkey(k["fact"])) not in know:
                iss.append(issue("KNOWLEDGE_BEFORE_GAIN", "error", f"{k['char_id']} acts on '{k['fact']}' in {sid} before learning it", [sid], [k["char_id"]]))
        # ---- injuries
        prev, mention = dict(inj), {i["char_id"] for i in sc["injuries"]}
        for i in sc["injuries"]:
            if i["status"] == "ongoing" and i["char_id"] not in prev:
                iss.append(issue("INJURY_NO_ORIGIN", "warning", f"{i['char_id']} has ongoing '{i['injury']}' in {sid} with no earlier onset", [sid], [i["char_id"]]))
            if i["status"] == "healed": inj.pop(i["char_id"], None)
            else: inj[i["char_id"]] = {"injury": i["injury"], "scene": sid}
        for cid in present:
            if cid in prev and cid not in mention:
                iss.append(issue("INJURY_DROPPED", "warning", f"{cid} had '{prev[cid]['injury']}' (from {prev[cid]['scene']}) but it is not shown in {sid}", [prev[cid]["scene"], sid], [cid]))
    return iss

def run_all_checks(project):
    iss = check_extraction_integrity(project) + project.state["norm_issues"] + check_duplicates(project.state) + check_continuity(project.state)
    order = {"error": 0, "warning": 1, "info": 2}
    iss.sort(key=lambda i: (order[i["severity"]], i["code"]))
    for n, i in enumerate(iss, 1): i["id"] = f"ISS{n:02d}"
    project.issues = iss
    return iss

def issues_table(issues):
    return pd.DataFrame([{"id": i["id"], "severity": i["severity"], "code": i["code"], "affected_scenes": ",".join(i["scenes"]), "message": i["message"], "fix_hint": i["fix_hint"]} for i in issues]) if issues else pd.DataFrame(columns=["id", "severity", "code", "affected_scenes", "message", "fix_hint"])

### 6.3 Run the checks on the sample — **3 deliberate contradictions** should be detected


In [ ]:
run_all_checks(P)
display(issues_table(P.issues))
assert {"PROP_TELEPORT", "COSTUME_CHANGE_UNEXPLAINED"} <= {i["code"] for i in P.issues}, "expected sample traps not detected"

---
# STEP 7 — Human review: correct, merge, rename, approve

### 7.1 Correction tools (har edit logged + sirf affected outputs invalid hote hain)
Corrections are applied to **raw + overrides**, then `normalize()` runs again, keeping the state reproducible.


In [ ]:
def refresh(project):
    normalize(project); run_all_checks(project); project.save()

def patch_scene(project, number, field, value, reason=""):
    s = project.raw["scenes"][number - 1]
    before = copy.deepcopy(s.get(field)); s[field] = value
    project.log("patch_scene", scene=number, field=field, before=before, after=value, reason=reason)
    refresh(project)

def _registry_of(st, eid):
    for kind in ("chars", "props", "locs"):
        if eid in st[kind]: return kind
    raise KeyError(f"{eid} not found (merge works for characters, props, locations)")

def merge_entities(project, keep_id, drop_id):
    st = project.state; kind = _registry_of(st, keep_id)
    keep, drop = st[kind][keep_id], st[kind][drop_id]
    for n in [drop["name"]] + drop["aliases"]: project.alias_overrides[nkey(n)] = keep["name"]
    project.log("merge", keep=keep_id, drop=drop_id, aliases_moved=[drop["name"]] + drop["aliases"])
    refresh(project)

def merge_costumes(project, keep_id, drop_id):
    st = project.state; keep, drop = st["costs"][keep_id], st["costs"][drop_id]
    assert keep["char_id"] == drop["char_id"], "costumes belong to different characters"
    idx = char_index(st)
    for sc in project.raw["scenes"]:
        if f"SC{sc['number']:02d}" in drop["scenes"]:
            for m in list(sc.get("costumes", {})):
                if resolve_speaker(idx, m) == drop["char_id"]: sc["costumes"][m] = keep["description"]
    project.log("merge_costume", keep=keep_id, drop=drop_id); refresh(project)

def edit_entity(project, eid, **fields):
    project.entity_edits.setdefault(eid, {}).update(fields)
    project.log("edit_entity", id=eid, fields=fields); refresh(project)

def rename_entity(project, eid, new_name): edit_entity(project, eid, name=new_name)
print("review tools ready: patch_scene, merge_entities, merge_costumes, edit_entity, rename_entity")

### 7.2 Hash-based approval gates
Gate **content-hash** se bandha hai: approve karne ke baad agar upstream data badla → approval **automatically invalid**. Isse "approve ke baad chupke se change" nahi ho sakta, aur expensive image generation hamesha approved state par hi chalta hai.


In [ ]:
class ApprovalBlocked(Exception): pass

def gate_hash(project, gate):
    st = project.state
    ext = sha({"reg": {k: st[k] for k in ("chars", "locs", "props", "costs")},
               "scenes": [{k: v for k, v in s.items() if k != "source_text"} for s in st["scenes"]]})
    if gate == "extraction": return ext
    if not project.plan: return None
    if gate == "plan": return sha({"plan": {k: v for k, v in project.plan.items() if k != "costumes"}, "up": ext})
    if gate == "costumes": return sha({"costumes": project.plan["costumes"], "up": ext})

def is_approved(project, gate):
    a = project.approvals.get(gate)
    return bool(a) and a["hash"] == gate_hash(project, gate)

def approve(project, gate, by="user", waive=()):
    if gate == "extraction":
        blocking = [i for i in project.issues if i["severity"] == "error" and i["code"] not in waive]
        if blocking: raise ApprovalBlocked(f"{len(blocking)} blocking issue(s): {[i['id'] + ':' + i['code'] for i in blocking]}")
    elif gate == "plan":
        if not is_approved(project, "extraction"): raise ApprovalBlocked("approve 'extraction' first")
        errs = [i for i in validate_plan(project) if i["severity"] == "error"]
        if errs: raise ApprovalBlocked(f"plan has errors: {[e['message'] for e in errs]}")
        if any(d.get("status") == "rejected" for d in project.plan["decisions"]): raise ApprovalBlocked("a plan decision is marked 'rejected' - revise it first")
    elif gate == "costumes":
        if not is_approved(project, "plan"): raise ApprovalBlocked("approve 'plan' first")
    project.approvals[gate] = {"hash": gate_hash(project, gate), "by": by, "ts": utcnow(), "waived": list(waive)}
    project.log("approve", gate=gate, by=by, waived=list(waive)); project.save()

def require(project, *gates):
    missing = [g for g in gates if not is_approved(project, g)]
    if missing: raise ApprovalBlocked(f"Not approved (or invalidated by later edits): {missing}")

def approval_status(project):
    return pd.DataFrame([{"gate": g, "approved": is_approved(project, g), "by": project.approvals.get(g, {}).get("by", ""), "at": project.approvals.get(g, {}).get("ts", "")} for g in Project.GATES])

### 7.3 Driver — gate **blocks** while contradictions exist, user fixes them, then approves
(Assignment demo item: *"One contradiction detected before visual generation"* aur *"user corrects an extraction and regenerates only affected outputs"*.)


In [ ]:
# (a) Approval is BLOCKED while errors exist
try:
    approve(P, "extraction")
except ApprovalBlocked as e:
    print("BLOCKED as expected ->", e)

# (b) Human fixes, with a reason logged for traceability
sc2 = P.raw["scenes"][1]
patch_scene(P, 2, "transfers", sc2["transfers"] + [{"prop": "letter", "from": "Karan", "to": "Amar"}],
            reason="Source omits Karan handing the letter back; added a hand-back at the end of Scene 2")
patch_scene(P, 3, "costume_change_reason", {"The Elder Son": "bathed and changed into clean clothes before facing his father"},
            reason="Costume change needs a motivation; bathing before the confession is believable")

print("\nIssues after fixes:")
display(issues_table(P.issues))
print("\nCostumes now:"); display(costume_table(P.state))
approve(P, "extraction")
display(approval_status(P))

---
# STEP 8 — Cultural adaptation **plan** (decide first, rewrite later)

Plan screenplay se *pehle* banta hai aur user ko dikhta hai. Isme: culture-specific **decisions + reasons + confidence**, naam/kinship mapping, character & **costume bible specs**, locations, props, aur **uncertainty register**. Har later asset `D01…` decision IDs se link hota hai.

### 8.1 Plan schema + prompt


In [ ]:
PLAN_SCHEMA = """{
 "culture_summary": str, "period": str,
 "decisions": [{"id": "D01", "dimension": "Verbal|Non-verbal|Characters|Visual world|Story world", "source": str, "adapted": str,
                "why": str, "scenes": ["SC01"], "confidence": "high|medium|low", "status": "proposed"}],
 "uncertainties": [{"id": "U01", "question": str, "assumption": str, "confidence": "low|medium"}],
 "names": {CHAR_ID: {"adapted_name": str, "short": str, "native": str (in output script), "addresses": {CHAR_ID: how this person addresses that one}}},
 "characters": {CHAR_ID: {"physical": str, "grooming": str, "personality_cues": str, "speech": str, "gesture_habits": str}},
 "costumes": {COST_ID: {"label": str, "garments": str, "fabrics": str, "colours": str, "colour_hex": [hex], "footwear": str,
                        "jewellery": str, "headwear": str, "grooming": str, "change_reason": str}},
 "locations": {LOC_ID: {"adapted_name": str, "description": str, "architecture": str, "palette_hex": [hex], "dressing": [str], "sound": [str], "animals": [str]}},
 "props": {PROP_ID: {"adapted_name": str, "description": str}},
 "preservation": [{"scene_id": "SC01", "purpose_kept": true, "note": str}]
}"""

def plan_system(sel):
    return (f"You are a cultural adaptation consultant and production designer for {sel['dialect']} ({sel['culture']}), region: {sel['region']}, "
            f"setting: {sel['setting']}, period: {sel['period']}. Output ONLY JSON. Rules: (1) Preserve each scene's dramatic purpose, character "
            "relationships and emotional arc; every transfer of a prop must still be possible. (2) Adapt kinship terms, idioms, humour, gesture, seating, food, "
            "dress, architecture, transport, sound - not just vocabulary. (3) Stay inside THIS dialect and region; never borrow traditions from neighbouring or "
            "unrelated cultures. (4) When unsure, say so: confidence 'low' + an entry in 'uncertainties'. (5) Cover EVERY canonical ID given.")

def compact_state(project):
    st = project.state
    return {"characters": [{k: c.get(k) for k in ("id", "name", "age", "role", "relationships", "personality", "emotional_state", "scenes")} for c in st["chars"].values()],
            "costumes": list(st["costs"].values()), "locations": list(st["locs"].values()), "props": list(st["props"].values()),
            "scenes": [{k: s.get(k) for k in ("scene_id", "time", "day", "location_id", "characters", "costume_ids", "props_in", "transfers", "knowledge_gain",
                                              "emotional_change", "production", "summary", "dramatic_purpose", "source_text")} for s in st["scenes"]]}

### 8.2 Bundled sample plan — **Majhi, rural Majha, Gurmukhi** (offline demo)
> ⚠️ Sample plan/dialogue *illustrative* hai aur native-speaker review ke baad hi "final" maana jaye — yeh `uncertainties` mein bhi likha hai. Live mode mein Claude yeh plan banata hai.


In [ ]:
SAMPLE_PLAN = {
 "culture_summary": "A Jat Sikh farming household of rural Majha (Amritsar-Gurdaspur belt). Restrained patriarch, food-as-love mother, "
                    "izzat/'pind wale ki kehnge' (what the village will say) as quiet social pressure. Emotion is shown through objects and seating, not speeches.",
 "period": "Contemporary, deep winter (Poh-Magh): wheat sown, mustard in flower, morning mist, chulha lit at night.",
 "decisions": [
  {"id": "D01", "dimension": "Verbal", "source": "Mother: 'You've become thin. Eat.'",
   "adapted": "Bebe uses the Majhi idiom 'sukk ke teela ho gya' (dried to a matchstick) and calls him 'putt'; the command is softened into 'beh ja, roti kha'.",
   "why": "Food is the love language; the idiom carries worry + affection, which a literal translation would flatten.", "scenes": ["SC01"], "confidence": "high", "status": "proposed"},
  {"id": "D02", "dimension": "Non-verbal", "source": "Amar enters and sits.",
   "adapted": "Jeeta bends to touch Bebe's feet (pairi hath); she places her palm on his head. No hug.",
   "why": "Greeting an elder after absence is a touch-the-feet blessing exchange; hugging between mother and grown son is less typical in a rural Majhi home.", "scenes": ["SC01"], "confidence": "medium", "status": "proposed"},
  {"id": "D03", "dimension": "Characters", "source": "Amar / Mother / Father / Karan",
   "adapted": "Amar -> Amarjit Singh 'Jeeta'; Mother -> Bebe; Father -> Bapu ji; Karan -> Karanbir 'Kaka' who calls his brother 'Veere'.",
   "why": "Pet names and kinship address define relationships in Majhi households; Jat Sikh naming with Singh/Kaur.", "scenes": ["SC01", "SC02", "SC03", "SC04"], "confidence": "medium", "status": "proposed"},
  {"id": "D04", "dimension": "Visual world", "source": "A pot on a stove; 'Eat.'",
   "adapted": "Wood-fired mud chulha; makki di roti with sarson da saag and white butter on a steel thali; lassi/tea in steel glasses.",
   "why": "Seasonal (winter) signature meal of Majha; roots the kitchen scene without dialogue.", "scenes": ["SC01"], "confidence": "high", "status": "proposed"},
  {"id": "D05", "dimension": "Visual world", "source": "Mist over crops; field path; tractor",
   "adapted": "Wheat fields with yellow sarson blossoms, tahli (shisham) trees, a tubewell kotha, a Swaraj tractor.",
   "why": "Landscape and machinery recognisable as Punjab's plains in winter.", "scenes": ["SC02"], "confidence": "high", "status": "proposed"},
  {"id": "D06", "dimension": "Story world", "source": "Father pats the cot and says 'Sit.'",
   "adapted": "Bapu says only 'Beh ja' and pats the manji; Jeeta sits at the paindi (foot end) out of respect - acceptance is shown by seating, not comfort-talk.",
   "why": "A Majhi patriarch rarely verbalises forgiveness; the invitation to sit on his manji is the emotional turn. Same arc (admission -> acceptance), culturally exact expression.", "scenes": ["SC03"], "confidence": "medium", "status": "proposed"},
  {"id": "D07", "dimension": "Story world", "source": "Amar hid the job loss for three months.",
   "adapted": "Adds the motive 'pind wale ki kehnge' (what will the villagers say) to Jeeta's confession.",
   "why": "Community influence/izzat makes the secrecy believable without changing his motivation (shame).", "scenes": ["SC03"], "confidence": "high", "status": "proposed"},
  {"id": "D08", "dimension": "Visual world", "source": "Bus stop; bus; bicycle; wristwatch",
   "adapted": "Village adda under a tahli tree near the gurdwara's Nishan Sahib; a PRTC bus; Kaka on a cycle; Bapu ties his old HMT-style watch on Jeeta's wrist.",
   "why": "Local transport and a heirloom-watch handover = blessing and trust.", "scenes": ["SC04"], "confidence": "medium", "status": "proposed"},
  {"id": "D09", "dimension": "Story world", "source": "Silence / atmosphere",
   "adapted": "Gurbani from the gurdwara loudspeaker at amrit vela, tubewell hum, buffalo bells, sparrows.",
   "why": "Sound-world of a Majha village morning; carries the culture even with dialogue muted.", "scenes": ["SC02", "SC03"], "confidence": "high", "status": "proposed"},
  {"id": "D10", "dimension": "Characters", "source": "Amar's 'city' look",
   "adapted": "Jeeta wears a black patka and steel kada even in city clothes; after bathing (SC03) he switches to clean white kurta-pyjama - a respectful dress to face his father.",
   "why": "Gives the unexplained shirt change a believable cultural motivation.", "scenes": ["SC03", "SC04"], "confidence": "medium", "status": "proposed"}],
 "uncertainties": [
  {"id": "U01", "question": "Is the household Sikh (patka/pagri/kada) or Hindu/other?", "assumption": "Jat Sikh farming family (majority in rural Majha).", "confidence": "medium"},
  {"id": "U02", "question": "Is the father addressed as 'Bapu ji', 'Pita ji', or 'Babaji' in this family?", "assumption": "'Bapu ji' (Bapu in narration).", "confidence": "medium"},
  {"id": "U03", "question": "Gurmukhi spellings and idioms are not native-verified.", "assumption": "Standard Punjabi is Majhi-based, so risk is low but non-zero; needs native review.", "confidence": "medium"}],
 "names": {
  "CHAR_AMAR": {"adapted_name": "Amarjit Singh 'Jeeta'", "short": "Jeeta", "native": "ਜੀਤਾ", "addresses": {"CHAR_MOTHER": "Bebe", "CHAR_FATHER": "Bapu ji", "CHAR_KARAN": "Kaka"}},
  "CHAR_MOTHER": {"adapted_name": "Bebe (Gurmeet Kaur)", "short": "Bebe", "native": "ਬੇਬੇ", "addresses": {"CHAR_AMAR": "putt", "CHAR_FATHER": "(indirect, 'tera Bapu')"}},
  "CHAR_FATHER": {"adapted_name": "Bapu ji (Harbans Singh)", "short": "Bapu", "native": "ਬਾਪੂ", "addresses": {"CHAR_AMAR": "putt", "CHAR_KARAN": "Kaka"}},
  "CHAR_KARAN": {"adapted_name": "Karanbir Singh 'Kaka'", "short": "Kaka", "native": "ਕਾਕਾ", "addresses": {"CHAR_AMAR": "Veere"}}},
 "characters": {
  "CHAR_AMAR": {"physical": "28-year-old Jat Sikh man, lean, about 5'10\", wheatish complexion, sharp jaw, trimmed black beard, tired deep-set eyes",
                "grooming": "trimmed beard, hair tied under a black patka", "personality_cues": "avoids eye contact with elders, shoulders slightly hunched, fiddles with kada",
                "speech": "soft, short answers, drops into Punjabi-English mix only with Kaka", "gesture_habits": "touches elders' feet, looks down when ashamed"},
  "CHAR_MOTHER": {"physical": "55-year-old woman, medium height, round face, wheatish, deep smile lines, grey streaks in an oiled plait",
                  "grooming": "hair in plait covered by dupatta, small gold koka (nose stud) and gold studs", "personality_cues": "always busy with her hands, watchful eyes",
                  "speech": "indirect, affectionate, repeats 'putt'", "gesture_habits": "serves food before asking, palm on head as blessing"},
  "CHAR_FATHER": {"physical": "60-year-old farmer, broad-shouldered, weathered sun-tanned skin, deep wrinkles, thick white beard neatly tied, strong forearms",
                  "grooming": "white beard (dhari) tied up, off-white pagri", "personality_cues": "long silences, stares at the fields, rarely smiles",
                  "speech": "few words, proverb-like, calm", "gesture_habits": "wipes hands on parna, pats the manji to invite"},
  "CHAR_KARAN": {"physical": "19-year-old slim, lanky young man, wheatish, light stubble, short hair", "grooming": "short hair under a black woollen cap",
                 "personality_cues": "restless, grins easily, curious", "speech": "playful, uses 'Veere'", "gesture_habits": "leans on his cycle, bounces on his heels"}},
 "costumes": {
  "COST_AMAR_01": {"label": "City return look", "garments": "grey cotton shirt, faded blue jeans, black woollen chaddar over shoulder", "fabrics": "cotton, denim, wool",
                   "colours": "grey, faded blue, black", "colour_hex": ["#6b6b6b", "#3b5a82", "#111111"], "footwear": "white sneakers, dusty", "jewellery": "steel kada on right wrist",
                   "headwear": "black patka", "grooming": "trimmed beard", "change_reason": ""},
  "COST_AMAR_02": {"label": "Clean kurta-pyjama after bath", "garments": "white cotton kurta and pyjama, black woollen chaddar", "fabrics": "cotton, wool",
                   "colours": "white, black", "colour_hex": ["#f2f0ea", "#111111"], "footwear": "bare feet in courtyard; plain leather chappals at the bus stop",
                   "jewellery": "steel kada on right wrist", "headwear": "black patka", "grooming": "trimmed beard, damp hair", "change_reason": "Bathed and dressed respectfully to face his father (D10)"},
  "COST_MOTHER_01": {"label": "Winter home wear", "garments": "dull green cotton salwar-kameez, phulkari shawl, dupatta over head", "fabrics": "cotton, wool",
                     "colours": "dull green, red-orange phulkari", "colour_hex": ["#5d7a4e", "#c4572a"], "footwear": "rubber chappals", "jewellery": "gold koka, gold studs, glass-and-steel bangles",
                     "headwear": "dupatta", "grooming": "oiled plait", "change_reason": ""},
  "COST_FATHER_01": {"label": "Farmer's winter wear", "garments": "off-white kurta, chadra (tied lower wrap), woollen sweater, grey shawl", "fabrics": "cotton, wool",
                     "colours": "off-white, brown, grey", "colour_hex": ["#e8e3d3", "#6b4a2d", "#7d7d7d"], "footwear": "Punjabi jutti", "jewellery": "steel kada",
                     "headwear": "off-white pagri", "grooming": "white tied beard", "change_reason": ""},
  "COST_KARAN_01": {"label": "Village youth winter", "garments": "blue checked shirt, jeans, hooded sweatshirt", "fabrics": "cotton, fleece",
                    "colours": "blue-check, indigo, black", "colour_hex": ["#2b4f81", "#1d2a44", "#111111"], "footwear": "canvas shoes", "jewellery": "steel kada",
                    "headwear": "black woollen cap", "grooming": "light stubble", "change_reason": ""}},
 "locations": {
  "LOC_FAMILY_KITCHEN": {"adapted_name": "Rasoi (kitchen) of a Majha farmhouse", "description": "Smoke-darkened brick kitchen with a wood-fired mud chulha, low wooden piri to sit on, steel thalis and katoris on a wall shelf, a single bulb",
                         "architecture": "pukka brick, lime-washed wall, low door", "palette_hex": ["#3a2a1e", "#d9a066", "#8b5a2b"], "dressing": ["chulha", "piri", "steel thali", "brass parat", "calendar of a gurdwara"], "sound": ["bulb hum", "crackle of fire"], "animals": []},
  "LOC_FIELD_PATH": {"adapted_name": "Vatt (field bund) near the village", "description": "Raised earthen bund between wheat fields, yellow sarson blossoms, tahli trees, a tubewell kotha in the distance, thick winter mist",
                     "architecture": "earthen bund, small brick tubewell room", "palette_hex": ["#9fb4a0", "#e3c75a", "#cfd6d4"], "dressing": ["tubewell kotha", "irrigation channel", "tahli tree"], "sound": ["Gurbani from distant gurdwara speaker", "tractor rumble"], "animals": ["sparrows"]},
  "LOC_COURTYARD": {"adapted_name": "Vehra (courtyard)", "description": "Open brick-paved vehra at dusk with a neem tree, a manji with a hand-woven rope bed, buffaloes tied at the far side, a parked Swaraj tractor, red-lit sky",
                    "architecture": "brick-paved open yard, verandah with arched pillars", "palette_hex": ["#b5602f", "#5b3a29", "#e6b27a"], "dressing": ["manji", "neem tree", "buffalo pen", "stack of dung cakes", "steel water drum"], "sound": ["buffalo bells", "distant Gurbani", "evening birds"], "animals": ["two buffaloes"]},
  "LOC_VILLAGE_BUS_STOP": {"adapted_name": "Village adda (bus stop)", "description": "Roadside adda under a tahli tree, a rusty tin shed, a PRTC bus idling, Nishan Sahib flag of a gurdwara in the background, morning light",
                           "architecture": "roadside tin shed, brick pillars", "palette_hex": ["#7a8c99", "#d6b86a", "#a8472a"], "dressing": ["PRTC bus", "tin shed", "bicycle", "Nishan Sahib flag", "tea stall"], "sound": ["bus engine idling", "horn"], "animals": ["stray dog"]}},
 "props": {
  "PROP_LETTER": {"adapted_name": "Chitthi (job-termination letter in a brown envelope)", "description": "Folded letter in a creased brown envelope marked A. Singh"},
  "PROP_DUFFEL_BAG": {"adapted_name": "Jhola (worn duffel bag)", "description": "Faded navy duffel bag with a broken zip-pull"},
  "PROP_WRISTWATCH": {"adapted_name": "Bapu's old ghadi (wristwatch)", "description": "Old steel wristwatch with a cracked leather strap, HMT-style round dial"}},
 "preservation": [
  {"scene_id": "SC01", "purpose_kept": True, "note": "Secret established; love shown via food."},
  {"scene_id": "SC02", "purpose_kept": True, "note": "Secret spreads to Kaka; letter handed over and returned."},
  {"scene_id": "SC03", "purpose_kept": True, "note": "Admission -> acceptance; shown by seating instead of comfort-talk (D06)."},
  {"scene_id": "SC04", "purpose_kept": True, "note": "Watch handover = trust; Jeeta stays."}]}

def skeleton_plan(project):
    st = project.state
    return {"culture_summary": "TODO - generate with a live model or fill manually", "period": project.selection["period"], "decisions": [], "uncertainties": [
                {"id": "U00", "question": "Plan is an empty skeleton (no live LLM, no bundled sample for this culture).", "assumption": "User must supply cultural decisions.", "confidence": "low"}],
            "names": {c: {"adapted_name": v["name"], "short": v["name"], "native": v["name"], "addresses": {}} for c, v in st["chars"].items()},
            "characters": {c: {"physical": "", "grooming": "", "personality_cues": v.get("personality", ""), "speech": "", "gesture_habits": ""} for c, v in st["chars"].items()},
            "costumes": {k: {"label": k, "garments": v["description"], "fabrics": "", "colours": "", "colour_hex": [], "footwear": "", "jewellery": "", "headwear": "", "grooming": "", "change_reason": ""} for k, v in st["costs"].items()},
            "locations": {k: {"adapted_name": v["name"], "description": "", "architecture": "", "palette_hex": [], "dressing": [], "sound": [], "animals": []} for k, v in st["locs"].items()},
            "props": {k: {"adapted_name": v["name"], "description": ""} for k, v in st["props"].items()},
            "preservation": [{"scene_id": s["scene_id"], "purpose_kept": True, "note": ""} for s in st["scenes"]]}

### 8.3 `build_plan()` + `validate_plan()` (coverage, story preservation, cultural-mixing lint, uncertainty register)


In [ ]:
def plan_text(plan):
    return json.dumps(plan, ensure_ascii=False)

def build_plan(project):
    require(project, "extraction")
    mock_ok = is_sample(project) and culture_key(project.selection) == "majhi"
    user = (f"Canonical production state (use these IDs exactly):\n{json.dumps(compact_state(project), ensure_ascii=False)}\n\nReturn JSON with schema:\n{PLAN_SCHEMA}")
    guard = mixing_lint(project.selection, user)
    assert not guard, f"prompt leaks other-culture markers: {guard}"
    plan = llm_json("cultural_plan", plan_system(project.selection), user, max_tokens=12000,
                    mock=lambda: copy.deepcopy(SAMPLE_PLAN) if mock_ok else skeleton_plan(project))
    project.plan = plan; project.log("build_plan", decisions=len(plan.get("decisions", [])))
    project.issues_plan = validate_plan(project); project.save()
    return plan

def validate_plan(project):
    st, pl, iss = project.state, project.plan, []
    for reg, key, label in ((st["chars"], "names", "names"), (st["chars"], "characters", "characters"), (st["costs"], "costumes", "costumes"),
                            (st["locs"], "locations", "locations"), (st["props"], "props", "props")):
        for eid in reg:
            if eid not in pl.get(key, {}): iss.append(issue("PLAN_MISSING_" + label.upper(), "error", f"Plan has no '{key}' entry for {eid}", entities=[eid]))
    sids = {s["scene_id"] for s in st["scenes"]}
    pres = {p["scene_id"]: p for p in pl.get("preservation", [])}
    for sid in sids:
        if sid not in pres: iss.append(issue("PLAN_NO_PRESERVATION", "error", f"No story-preservation note for {sid}", [sid]))
        elif not pres[sid].get("purpose_kept"): iss.append(issue("PLAN_STORY_CHANGED", "error", f"{sid}: plan says dramatic purpose NOT kept - justify or revise", [sid]))
    for d in pl.get("decisions", []):
        bad = [s for s in d.get("scenes", []) if s not in sids]
        if bad: iss.append(issue("PLAN_BAD_SCENE_REF", "error", f"{d['id']} references unknown scenes {bad}"))
        if d.get("confidence") == "low": iss.append(issue("PLAN_LOW_CONFIDENCE", "warning", f"{d['id']} is low confidence - review with a native speaker", d.get("scenes", [])))
    for u in pl.get("uncertainties", []): iss.append(issue("PLAN_UNCERTAINTY", "info", f"{u['id']}: {u['question']} (assumed: {u['assumption']})"))
    mix = mixing_lint(project.selection, plan_text(pl))
    if mix: iss.append(issue("PLAN_CULTURAL_MIXING", "warning", f"Possible cross-culture markers in plan: {mix}"))
    for cid, c in pl.get("costumes", {}).items():
        if cid in st["costs"] and not c.get("garments"): iss.append(issue("PLAN_EMPTY_COSTUME", "error", f"{cid} has no garments in plan", entities=[cid]))
    return iss

def edit_plan(project, path, value, reason=""):
    """edit_plan(P, ['costumes','COST_AMAR_01','colours'], 'charcoal, indigo') -> invalidates plan/costume approval by hash."""
    node = project.plan
    for k in path[:-1]: node = node[k]
    before = copy.deepcopy(node[path[-1]]); node[path[-1]] = value
    project.log("edit_plan", path=path, before=before, after=value, reason=reason)
    project.issues_plan = validate_plan(project); project.save()

def set_decision_status(project, did, status):
    d = next(d for d in project.plan["decisions"] if d["id"] == did); d["status"] = status
    project.log("decision_status", id=did, status=status)

def plan_table(project):
    return pd.DataFrame([{"id": d["id"], "dimension": d["dimension"], "scenes": ",".join(d["scenes"]), "source": d["source"], "adapted": d["adapted"],
                          "why": d["why"], "confidence": d["confidence"], "status": d.get("status", "proposed")} for d in project.plan["decisions"]])

### 8.4 Driver — build the plan, review it, and approve the **plan** and **costume plan** (Gates 2 & 3)


In [ ]:
build_plan(P)
pd.set_option("display.max_colwidth", 90)
display(plan_table(P))
display(issues_table([i for i in validate_plan(P) if i["severity"] != "info"]))
print("Uncertainty register:"); display(pd.DataFrame(P.plan["uncertainties"]))
display(pd.DataFrame([{"costume": k, "label": v["label"], "garments": v["garments"], "headwear": v["headwear"], "jewellery": v["jewellery"], "footwear": v["footwear"]} for k, v in P.plan["costumes"].items()]))

for d in P.plan["decisions"]: set_decision_status(P, d["id"], "accepted")
approve(P, "plan"); approve(P, "costumes")
display(approval_status(P))

---
# STEP 9 — Screenplay adaptation (rewrite from the **approved** plan, fully traceable)

Rewriting runs only when `extraction` + `plan` are approved. Each scene is a separate call; every line/action includes `plan_refs` (decision IDs) and `source_scene` — this provides **traceability**.

### 9.1 Output schema + prompt


In [ ]:
ADAPT_SCHEMA = """{
 "scene_id": "SC01", "source_scene": "SC01", "heading": str (in output script), "heading_roman": str,
 "beats": [ {"type": "action", "text": str (output script), "en": str, "plan_refs": ["D04"]},
            {"type": "dialogue", "char_id": CHAR_ID, "speaker": str, "paren": str, "line": str (output script), "roman": str, "en": str, "plan_refs": ["D01"]} ],
 "events": {"transfers": [{"prop_id": PROP_ID, "from": CHAR_ID, "to": CHAR_ID}]},
 "changes": [{"source": str, "adapted": str, "why": str, "plan_refs": ["D01"]}]
}"""

def adapt_system(sel):
    return (f"You are a screenwriter native to {sel['dialect']} ({sel['culture']}), region {sel['region']}, {sel['setting']} setting. Write ONLY JSON. "
            f"Dialogue and action in {sel['script']} script, plus a Roman transliteration and an English gloss for reviewers. Rewrite for dialect, rhythm, humour, "
            "kinship terms, respect and gesture - never word-for-word. Keep the dramatic purpose, the emotional arc and every prop transfer of the source scene. "
            "Use ONLY the approved plan (names, costumes, places, props). Do not import traditions from other regions. Cite plan decision IDs in plan_refs.")

def prompt_adapt(project, sc, prev_summary):
    plan = project.plan
    scene_plan = {"names": plan["names"], "decisions": [d for d in plan["decisions"] if sc["scene_id"] in d["scenes"]],
                  "location": plan["locations"].get(sc["location_id"]), "props": plan["props"],
                  "preservation": [p for p in plan["preservation"] if p["scene_id"] == sc["scene_id"]]}
    keep = {"scene_id": sc["scene_id"], "transfers": sc["transfers"], "characters": sc["characters"], "dramatic_purpose": sc["dramatic_purpose"]}
    return (f"APPROVED PLAN (this scene): {json.dumps(scene_plan, ensure_ascii=False)}\nMUST-KEEP FACTS: {json.dumps(keep, ensure_ascii=False)}\n"
            f"PREVIOUS ADAPTED SCENE (summary): {prev_summary}\n\nSOURCE SCENE:\n{sc['source_text']}\n\nReturn JSON:\n{ADAPT_SCHEMA}")

### 9.2 Bundled sample adaptation (Majhi, Gurmukhi)
Yeh mock data hai taaki notebook offline chale; live mode mein `prompt_adapt()` se Claude likhta hai. Roman + English gloss reviewers ke liye hain (native-speaker verification ke liye).


In [ ]:
def A(t, en, refs): return {"type": "action", "text": t, "en": en, "plan_refs": refs}
def D(cid, spk, line, roman, en, refs, paren=""): return {"type": "dialogue", "char_id": cid, "speaker": spk, "paren": paren, "line": line, "roman": roman, "en": en, "plan_refs": refs}

SAMPLE_ADAPTED = {
 "SC01": {"scene_id": "SC01", "source_scene": "SC01", "heading": "ਅੰਦਰ. ਘਰ ਦੀ ਰਸੋਈ – ਰਾਤ", "heading_roman": "Andar. Ghar di rasoi - raat",
  "beats": [
   A("ਚੁੱਲ੍ਹੇ ਦੀ ਅੱਗ ਮੱਧਮ ਬਲ ਰਹੀ ਏ। ਬੇਬੇ ਪਤੀਲੇ ਵਿੱਚ ਕੜਛੀ ਫੇਰਦੀ ਏ; ਕੋਲ ਮੱਕੀ ਦੀਆਂ ਰੋਟੀਆਂ ਤੇ ਸਰ੍ਹੋਂ ਦਾ ਸਾਗ ਤਿਆਰ ਏ।", "The chulha burns low. Bebe stirs the pot; makki rotis and sarson saag are ready.", ["D04"]),
   A("ਜੀਤਾ ਝੋਲਾ ਲਈ ਅੰਦਰ ਆਉਂਦਾ ਏ, ਬੇਬੇ ਦੇ ਪੈਰੀਂ ਹੱਥ ਲਾਉਂਦਾ ਏ; ਬੇਬੇ ਉਹਦੇ ਸਿਰ ਤੇ ਹੱਥ ਧਰਦੀ ਏ। ਜੇਬ ਵਿੱਚੋਂ ਚਿੱਠੀ ਦਾ ਕੋਨਾ ਦਿਸਦਾ ਏ।", "Jeeta enters with his bag, touches Bebe's feet; she places a palm on his head. A corner of the letter shows in his pocket.", ["D02"]),
   D("CHAR_MOTHER", "ਬੇਬੇ", "ਵੇ ਪੁੱਤ, ਤੂੰ ਤਾਂ ਸੁੱਕ ਕੇ ਤੀਲਾ ਹੋ ਗਿਆ ਏਂ। ਬਹਿ ਜਾ, ਰੋਟੀ ਖਾ।", "Ve putt, tu taan sukk ke teela ho gya en. Beh ja, roti kha.", "Son, you've dried up to a matchstick. Sit, eat.", ["D01", "D03"]),
   D("CHAR_AMAR", "ਜੀਤਾ", "ਬੇਬੇ, ਮੈਂ ਰਾਹ ਵਿੱਚ ਹੀ ਖਾ ਲਿਆ ਸੀ।", "Bebe, main raah vich hi kha liya si.", "Bebe, I ate on the way.", ["D03"]),
   A("ਬੇਬੇ ਫੇਰ ਵੀ ਸਟੀਲ ਦੀ ਥਾਲੀ ਵਿੱਚ ਰੋਟੀ ਪਰੋਸ ਦਿੰਦੀ ਏ, ਉੱਤੇ ਚਿੱਟਾ ਮੱਖਣ ਧਰਦੀ ਏ। ਬਾਪੂ ਬੂਹੇ ਵਿੱਚ ਖੜ੍ਹਾ ਪਰਨੇ ਨਾਲ ਹੱਥ ਪੂੰਝਦਾ ਏ ਤੇ ਬਿਨਾਂ ਬੋਲੇ ਮੁੜ ਜਾਂਦਾ ਏ।", "Bebe serves anyway on a steel thali with white butter. Bapu wipes his hands on his parna in the doorway and leaves without a word.", ["D04", "D06"]),
   D("CHAR_MOTHER", "ਬੇਬੇ", "ਤੇਰਾ ਬਾਪੂ ਤੇਰੀ ਉਡੀਕ ਵਿੱਚ ਸਾਰੀ ਰਾਤ ਜਾਗਦਾ ਰਿਹਾ ਏ।", "Tera Bapu teri udeek vich saari raat jaagda reha e.", "Your Bapu stayed awake all night waiting for you.", ["D03"], paren="ਹੌਲੀ ਜਿਹੇ"),
   A("ਜੀਤਾ ਚਿੱਠੀ ਨੂੰ ਜੇਬ ਵਿੱਚ ਹੋਰ ਡੂੰਘੀ ਧੱਕ ਲੈਂਦਾ ਏ।", "Jeeta pushes the letter deeper into his pocket.", [])],
  "events": {"transfers": []},
  "changes": [{"source": "'You've become thin. Eat.'", "adapted": "'sukk ke teela' idiom + 'putt' + 'beh ja'", "why": "Food-as-love; idiom carries worry and affection.", "plan_refs": ["D01"]},
              {"source": "Amar enters", "adapted": "Touches Bebe's feet; palm on head", "why": "Greeting an elder after absence.", "plan_refs": ["D02"]}]},
 "SC02": {"scene_id": "SC02", "source_scene": "SC02", "heading": "ਬਾਹਰ. ਖੇਤਾਂ ਦੀ ਵੱਟ – ਸਵੇਰ", "heading_roman": "Bahar. Kheta di vatt - saver",
  "beats": [
   A("ਧੁੰਦ ਵਿੱਚ ਕਣਕ ਦੇ ਖੇਤਾਂ ਦੀ ਵੱਟ ਤੇ ਜੀਤਾ ਤੇ ਕਾਕਾ ਤੁਰੇ ਜਾਂਦੇ ਨੇ; ਸਰ੍ਹੋਂ ਦੇ ਪੀਲੇ ਫੁੱਲ ਤੇ ਦੂਰੋਂ ਗੁਰਦੁਆਰੇ ਦੇ ਸਪੀਕਰ ਚੋਂ ਗੁਰਬਾਣੀ ਦੀ ਆਵਾਜ਼। ਕਾਕੇ ਦੀ ਨਜ਼ਰ ਚਿੱਠੀ ਤੇ ਪੈਂਦੀ ਏ।", "In the mist Jeeta and Kaka walk the bund of wheat fields; yellow sarson, Gurbani from the distant gurdwara speaker. Kaka notices the letter.", ["D05", "D09"]),
   D("CHAR_KARAN", "ਕਾਕਾ", "ਵੀਰੇ, ਏਹ ਕਾਗਜ਼ ਕਿਹੜਾ ਏ?", "Veere, eh kaagaz kehda e?", "Brother, what is that paper?", ["D03"]),
   D("CHAR_AMAR", "ਜੀਤਾ", "ਕੁਝ ਨਹੀਂ। ਸਾਂਭ ਕੇ ਰੱਖ ਲੈ, ਤੇ ਬਾਪੂ ਨੂੰ ਨਾ ਦੱਸੀਂ।", "Kujh nahi. Sambh ke rakh lai, te Bapu nu na dassin.", "Nothing. Keep it safe, and don't tell Bapu.", ["D03"]),
   A("ਜੀਤਾ ਚਿੱਠੀ ਕਾਕੇ ਨੂੰ ਫੜਾ ਦਿੰਦਾ ਏ; ਕਾਕਾ ਜੇਬ ਵਿੱਚ ਪਾ ਲੈਂਦਾ ਏ। ਨੇੜਿਓਂ ਸਵਰਾਜ ਟਰੈਕਟਰ ਲੰਘਦਾ ਏ, ਦੋਵੇਂ ਚੁੱਪ ਕਰ ਕੇ ਵੇਖਦੇ ਨੇ।", "Jeeta hands the letter to Kaka, who pockets it. A Swaraj tractor passes; they watch in silence.", ["D05"]),
   D("CHAR_KARAN", "ਕਾਕਾ", "ਤੂੰ ਮੁੜ ਸ਼ਹਿਰ ਜਾਣਾ ਏ?", "Tu mud shehar jana e?", "Are you going back to the city?", []),
   D("CHAR_AMAR", "ਜੀਤਾ", "ਪਤਾ ਨਹੀਂ, ਕਾਕਾ।", "Pata nahi, Kaka.", "I don't know, Kaka.", []),
   A("ਕਾਕਾ ਚਿੱਠੀ ਮੋੜ ਦਿੰਦਾ ਏ; ਜੀਤਾ ਉਹਨੂੰ ਝੱਟ ਜੇਬ ਵਿੱਚ ਲੁਕਾ ਲੈਂਦਾ ਏ।", "Kaka returns the letter; Jeeta quickly hides it in his pocket.", [])],
  "events": {"transfers": [{"prop_id": "PROP_LETTER", "from": "CHAR_AMAR", "to": "CHAR_KARAN"}, {"prop_id": "PROP_LETTER", "from": "CHAR_KARAN", "to": "CHAR_AMAR"}]},
  "changes": [{"source": "Mist over crops; tractor", "adapted": "Wheat bund, sarson blossoms, Swaraj tractor, Gurbani from the gurdwara speaker", "why": "Recognisable winter Majha soundscape/landscape.", "plan_refs": ["D05", "D09"]},
              {"source": "Karan: 'Are you going back?'", "adapted": "Kaka calls him 'Veere' and hands the letter back (continuity fix)", "why": "Kinship address; resolves the prop continuity gap found in review.", "plan_refs": ["D03"]}]},
 "SC03": {"scene_id": "SC03", "source_scene": "SC03", "heading": "ਬਾਹਰ. ਵਿਹੜਾ – ਸ਼ਾਮ", "heading_roman": "Bahar. Vehra - shaam",
  "beats": [
   A("ਸ਼ਾਮ ਦਾ ਵਿਹੜਾ: ਨਿੰਮ ਹੇਠ ਮੰਜੀ ਡਹੀ ਏ, ਪਿੱਛੇ ਮੱਝਾਂ ਬੱਝੀਆਂ ਨੇ। ਬਾਪੂ ਮੰਜੀ ਤੇ ਸਟੀਲ ਦੇ ਗਲਾਸ ਵਿੱਚ ਚਾਹ ਫੜੀ ਬੈਠਾ ਏ। ਨਹਾ ਕੇ ਚਿੱਟੇ ਕੁੜਤੇ-ਪਜਾਮੇ ਵਿੱਚ ਜੀਤਾ ਹੱਥ ਬੰਨ੍ਹੀ ਅੱਗੇ ਖੜ੍ਹਾ ਏ।", "Dusk in the vehra: a manji under the neem, buffaloes tied behind. Bapu sits with tea in a steel glass. Freshly bathed, in white kurta-pyjama, Jeeta stands before him, hands folded.", ["D06", "D10"]),
   D("CHAR_AMAR", "ਜੀਤਾ", "ਬਾਪੂ ਜੀ, ਮੇਰੀ ਨੌਕਰੀ ਤਿੰਨ ਮਹੀਨੇ ਪਹਿਲਾਂ ਹੀ ਛੁੱਟ ਗਈ ਸੀ।", "Bapu ji, meri naukri tinn mahine pehlan hi chhutt gayi si.", "Bapu ji, I lost my job three months ago.", ["D03"]),
   A("ਲੰਮੀ ਚੁੱਪ। ਬਾਪੂ ਗਲਾਸ ਥੱਲੇ ਰੱਖ ਦਿੰਦਾ ਏ ਤੇ ਦੂਰ ਖੇਤਾਂ ਵੱਲ ਵੇਖਦਾ ਏ।", "A long silence. Bapu sets the glass down and stares at the distant fields.", ["D06"]),
   D("CHAR_AMAR", "ਜੀਤਾ", "ਪਿੰਡ ਵਾਲੇ ਕੀ ਕਹਿਣਗੇ, ਇਹੀ ਸੋਚ ਕੇ ਚੁੱਪ ਰਿਹਾ।", "Pind wale ki kehnge, ehi soch ke chup reha.", "I stayed quiet thinking what the villagers would say.", ["D07"]),
   A("ਜੀਤਾ ਜੇਬ ਵਿੱਚੋਂ ਚਿੱਠੀ ਕੱਢ ਕੇ ਅੱਗੇ ਕਰਦਾ ਏ। ਬਾਪੂ ਐਨਕ ਲਾ ਕੇ ਹੌਲੀ-ਹੌਲੀ ਪੜ੍ਹਦਾ ਏ, ਤਹਿ ਕਰ ਕੇ ਆਪਣੀ ਜੇਬ ਵਿੱਚ ਰੱਖ ਲੈਂਦਾ ਏ, ਤੇ ਮੰਜੀ ਤੇ ਆਪਣੇ ਕੋਲ ਥਾਂ ਥਪਥਪਾਉਂਦਾ ਏ।", "Jeeta takes the letter from his pocket and offers it. Bapu reads slowly with his glasses, folds it into his own pocket, and pats the manji beside him.", ["D06"]),
   D("CHAR_FATHER", "ਬਾਪੂ", "ਬਹਿ ਜਾ।", "Beh ja.", "Sit.", ["D06"]),
   A("ਜੀਤਾ ਮੰਜੀ ਦੀ ਪੈਂਦੀ ਤੇ ਬਹਿ ਜਾਂਦਾ ਏ।", "Jeeta sits on the paindi (foot end) of the manji.", ["D06"]),
   D("CHAR_FATHER", "ਬਾਪੂ", "ਝੂਠ ਨਾਲ ਬੋਝ ਦੂਣਾ ਹੋ ਜਾਂਦਾ ਏ, ਪੁੱਤ। ਅਗਲੀ ਵਾਰ ਸਾਡੇ ਨਾਲ ਚੁੱਕੀਂ।", "Jhooth naal bojh doona ho janda e, putt. Agli vaar sade naal chukki.", "A lie doubles the load, son. Next time, carry it with us.", ["D06", "D03"])],
  "events": {"transfers": [{"prop_id": "PROP_LETTER", "from": "CHAR_AMAR", "to": "CHAR_FATHER"}]},
  "changes": [{"source": "Father pats the cot: 'Sit.'", "adapted": "Only 'Beh ja' + pat on the manji; Jeeta sits at the paindi", "why": "Acceptance via seating, not comfort-talk - same arc, culturally exact.", "plan_refs": ["D06"]},
              {"source": "(none)", "adapted": "'Pind wale ki kehnge' added to the confession", "why": "Community izzat explains the three-month secrecy.", "plan_refs": ["D07"]}]},
 "SC04": {"scene_id": "SC04", "source_scene": "SC04", "heading": "ਬਾਹਰ. ਪਿੰਡ ਦਾ ਬੱਸ ਅੱਡਾ – ਦਿਨ", "heading_roman": "Bahar. Pind da bus adda - din",
  "beats": [
   A("ਅਗਲੀ ਸਵੇਰ ਪਿੰਡ ਦੇ ਅੱਡੇ ਤੇ ਟਾਹਲੀ ਹੇਠ ਪੀਆਰਟੀਸੀ ਦੀ ਬੱਸ ਖੜ੍ਹੀ ਏ, ਇੰਜਣ ਚੱਲਦਾ ਏ। ਕਾਕਾ ਸਾਈਕਲ ਤੇ ਬੈਠਾ ਵੇਖ ਰਿਹਾ ਏ।", "Next morning at the village adda under a tahli tree a PRTC bus idles. Kaka watches from his bicycle.", ["D08"]),
   A("ਬਾਪੂ ਆਪਣੀ ਪੁਰਾਣੀ ਘੜੀ ਲਾਹ ਕੇ ਜੀਤੇ ਦੀ ਗੁੱਟ ਤੇ ਬੰਨ੍ਹ ਦਿੰਦਾ ਏ।", "Bapu unstraps his old watch and ties it on Jeeta's wrist.", ["D08"]),
   D("CHAR_FATHER", "ਬਾਪੂ", "ਖੇਤ ਉਡੀਕ ਲੈਣਗੇ, ਪਰ ਵਾਢੀ ਤੇ ਘਰ ਹੋਈਂ।", "Khet udeek lainge, par vaadhi te ghar hoin.", "The fields will wait, but be home for the harvest.", ["D08"]),
   A("ਜੀਤਾ ਬੱਸ ਵੱਲ ਵੇਖਦਾ ਏ, ਫੇਰ ਝੋਲਾ ਹੇਠਾਂ ਰੱਖ ਦਿੰਦਾ ਏ।", "Jeeta looks at the bus, then sets his bag down.", []),
   D("CHAR_AMAR", "ਜੀਤਾ", "ਮੈਂ ਕਿਤੇ ਨਹੀਂ ਜਾ ਰਿਹਾ, ਬਾਪੂ ਜੀ।", "Main kitte nahi ja riha, Bapu ji.", "I'm not going anywhere, Bapu ji.", ["D03"]),
   A("ਕਾਕਾ ਮੁਸਕਰਾ ਪੈਂਦਾ ਏ; ਬਾਪੂ ਇੱਕ ਵਾਰ ਹੌਲੀ ਜਿਹੀ ਸਿਰ ਹਿਲਾਉਂਦਾ ਏ।", "Kaka grins; Bapu nods once, slowly.", [])],
  "events": {"transfers": [{"prop_id": "PROP_WRISTWATCH", "from": "CHAR_FATHER", "to": "CHAR_AMAR"}]},
  "changes": [{"source": "Bus stop, bus", "adapted": "Village adda, PRTC bus, tahli tree, gurdwara flag", "why": "Local transport and place.", "plan_refs": ["D08"]},
              {"source": "'The farm will wait. Be here for the harvest.'", "adapted": "'Khet udeek lainge, par vaadhi te ghar hoin'", "why": "Vaadhi (harvest) is a Majha agricultural-calendar anchor.", "plan_refs": ["D08"]}]}}

### 9.3 `adapt_screenplay()` + **verification** (coverage, trace links, script check, prop events, mixing lint)
Verifier check karta hai: har source scene covered? har speaker canonical? `plan_refs` valid? text *selected script* mein hai? **source ke prop transfers adapted version mein survive kiye?** (plot-hole guard) — aur dialogue-turn count drift.


In [ ]:
def script_ratio(text, script):
    lo, hi = SCRIPTS[script]
    letters = [ch for ch in text if ch.isalpha()]
    return (sum(lo <= ord(ch) <= hi for ch in letters) / len(letters)) if letters else 1.0

def adapt_screenplay(project, only_scenes=None):
    require(project, "extraction", "plan")
    out = dict(project.adaptation or {})
    mock_ok = is_sample(project) and culture_key(project.selection) == "majhi"
    prev = "NONE"
    for sc in project.state["scenes"]:
        if only_scenes and sc["scene_id"] not in only_scenes:
            prev = out.get(sc["scene_id"], {}).get("heading_roman", prev); continue
        user = prompt_adapt(project, sc, prev)
        assert not mixing_lint(project.selection, user), "prompt leaks other-culture markers"
        try:
            rec = llm_json(f"adapt_{sc['scene_id']}", adapt_system(project.selection), user, max_tokens=6000,
                           mock=lambda sc=sc: copy.deepcopy(SAMPLE_ADAPTED[sc["scene_id"]]) if mock_ok else (_ for _ in ()).throw(LLMFailure("no mock rewrite for this culture/screenplay")))
            rec["plan_hash"] = project.approvals["plan"]["hash"]; out[sc["scene_id"]] = rec; prev = rec.get("heading_roman", prev)
        except LLMFailure as e:
            project.failures.append({"stage": "adaptation", "scene": sc["scene_id"], "error": str(e)})
    project.adaptation = out; project.log("adapt", scenes=list(out)); project.save()
    return out

def verify_adaptation(project):
    st, pl, ad, sel, iss = project.state, project.plan, project.adaptation or {}, project.selection, []
    ref_ids = {d["id"] for d in pl["decisions"]}
    for sc in st["scenes"]:
        sid = sc["scene_id"]; a = ad.get(sid)
        if not a: iss.append(issue("ADAPT_MISSING_SCENE", "error", f"No adapted version of {sid}", [sid])); continue
        speakers = [b for b in a["beats"] if b["type"] == "dialogue"]
        src_n = len(dialogue_turns(sc["source_text"]))
        if abs(len(speakers) - src_n) > 2: iss.append(issue("ADAPT_DIALOGUE_DRIFT", "warning", f"{sid}: {src_n} source turns vs {len(speakers)} adapted", [sid]))
        for b in a["beats"]:
            for r in b.get("plan_refs", []):
                if r not in ref_ids: iss.append(issue("ADAPT_BAD_REF", "error", f"{sid}: unknown plan ref {r}", [sid]))
            txt = b.get("line") or b.get("text") or ""
            if script_ratio(txt, sel["script"]) < 0.8: iss.append(issue("ADAPT_WRONG_SCRIPT", "error", f"{sid}: text not in {sel['script']}: '{txt[:30]}'", [sid]))
            if b["type"] == "dialogue" and b["char_id"] not in sc["characters"]: iss.append(issue("ADAPT_SPEAKER_ABSENT", "error", f"{sid}: {b['char_id']} speaks but is not in the scene", [sid], [b["char_id"]]))
        want = [(t["prop_id"], t["from"], t["to"]) for t in sc["transfers"]]
        got = [(t["prop_id"], t["from"], t["to"]) for t in a["events"]["transfers"]]
        if want != got: iss.append(issue("ADAPT_PLOT_HOLE", "error", f"{sid}: prop transfers changed. source={want} adapted={got}", [sid]))
        if not a.get("changes"): iss.append(issue("ADAPT_NO_EXPLANATION", "warning", f"{sid}: no change explanations recorded", [sid]))
        mix = mixing_lint(sel, plan_text(a))
        if mix: iss.append(issue("ADAPT_CULTURAL_MIXING", "warning", f"{sid}: foreign-culture markers {mix}", [sid]))
        if a.get("plan_hash") != project.approvals.get("plan", {}).get("hash"): iss.append(issue("ADAPT_STALE", "warning", f"{sid} was written from an older plan version - regenerate", [sid]))
    return iss

def screenplay_markdown(project, roman=True):
    pl, lines = project.plan, [f"# Adapted screenplay - {project.selection['dialect']} ({project.selection['region']}, {project.selection['setting']}) - {project.selection['script']}\n"]
    for sid, a in sorted(project.adaptation.items()):
        lines += [f"\n## {sid}: {a['heading']}  _({a['heading_roman']})_\n"]
        for b in a["beats"]:
            if b["type"] == "action": lines.append(f"*{b['text']}*  \n<sub>{b['en']} - refs {b['plan_refs']}</sub>\n")
            else:
                p = f" ({b['paren']})" if b["paren"] else ""
                lines.append(f"**{b['speaker']}**{p}: {b['line']}  \n<sub>{b['roman']} - \"{b['en']}\" - refs {b['plan_refs']}</sub>\n")
    return "\n".join(lines)

### 9.4 Driver — adapt, verify, and show **source vs adapted** side-by-side with explanations


In [ ]:
adapt_screenplay(P)
adapt_issues = verify_adaptation(P)
display(issues_table(adapt_issues) if adapt_issues else pd.DataFrame([{"result": "adaptation verification: no issues"}]))
assert not [i for i in adapt_issues if i["severity"] == "error"], adapt_issues

sid = "SC03"
print("SOURCE", sid, "\n" + P.state["scenes"][2]["source_text"])
print("\nADAPTED", sid, "-", P.adaptation[sid]["heading_roman"])
for b in P.adaptation[sid]["beats"]:
    print("  ", (b["speaker"] + ": " + b["line"] + "   | " + b["roman"]) if b["type"] == "dialogue" else ("[action] " + b["text"]))
print("\nWhy it changed:"); display(pd.DataFrame(P.adaptation[sid]["changes"]))

---
# STEP 10 — Visual production pack (character bible · costume bible · scene keyframes)

Image generation happens **only after approval**. Order: **characters → costumes → scenes**; each later image uses previously approved images as *references*.

### 10.1 Asset specs (reproducible) + dependency graph + **spec-hash staleness**
Har asset ka `spec` = model + prompt + negative prompt + seed + size + reference asset IDs. `effective_hash` = spec + **dependencies ke hashes** → ek costume badla to sirf uss costume ke aur uss costume wale scenes ke images stale; baaki nahi.


In [ ]:
STYLE = ("cinematic still, 35mm film look, natural light, photorealistic, documentary realism, shallow depth of field")
NEGATIVE = ("text, watermark, logo, extra fingers, deformed hands, duplicate person, western stereotype costume, costume party look, "
            "mixed regional dress, anachronistic modern fashion, caricature")
TIME_LIGHT = {"night": "single warm bulb and fire glow, deep shadows", "morning": "cold blue mist, soft low sun", "evening": "red-orange dusk, long shadows",
              "day": "clear soft winter daylight", "early morning": "pre-dawn blue light"}

def culture_header(sel):
    return f"{sel['dialect']} {sel['culture']} culture, {sel['region']}, {sel['setting']} setting, {sel['period']}"

def asset_ids(project):
    st = project.state
    return ([f"IMG_CHAR_{c}" for c in st["chars"]] + [f"IMG_COST_{c}" for c in st["costs"]] + [f"IMG_SCENE_{s['scene_id']}" for s in st["scenes"]])

def costume_phrase(c):
    return (f"{c['garments']}; fabrics {c['fabrics']}; colours {c['colours']}; footwear {c['footwear']}; jewellery {c['jewellery']}; "
            f"headwear {c['headwear']}; grooming {c['grooming']}")

def identity_lock(project, cid):
    ch, nm = project.plan["characters"][cid], project.plan["names"][cid]
    c = project.state["chars"][cid]
    return f"{nm['adapted_name']}, age {c.get('age')}: {ch['physical']}; {ch['grooming']}"

def build_specs(project):
    require(project, "extraction", "plan", "costumes")
    st, pl, sel, model = project.state, project.plan, project.selection, CFG["openai_image_model"] if CFG["image_backend"] == "openai" else "placeholder-renderer"
    hdr, specs = culture_header(sel), {}
    seed = int(sha(project.slug, 8), 16) % 10**6
    first_cost = {}
    for k, c in st["costs"].items(): first_cost.setdefault(c["char_id"], k)
    for cid in st["chars"]:                                  # one reference image per canonical character
        cost = pl["costumes"][first_cost[cid]]
        specs[f"IMG_CHAR_{cid}"] = {"kind": "character", "ref_id": cid, "model": model, "size": "1024x1536", "seed": seed, "deps": [],
            "prompt": f"{STYLE}. Full-body standing reference photo on a plain neutral wall, front view. {hdr}. Person: {identity_lock(project, cid)}. "
                      f"Wearing: {costume_phrase(cost)}. Posture cues: {pl['characters'][cid]['personality_cues']}.",
            "negative": NEGATIVE, "palette": cost["colour_hex"], "traces": {"source_scenes": st["chars"][cid]["scenes"], "plan": ["names", "characters"]}}
    for k, c in st["costs"].items():                         # each unique costume exactly once
        cid, cost = c["char_id"], pl["costumes"][k]
        specs[f"IMG_COST_{k}"] = {"kind": "costume", "ref_id": k, "model": model, "size": "1024x1536", "seed": seed, "deps": [f"IMG_CHAR_{cid}"],
            "prompt": f"{STYLE}. Costume sheet, full-body, front view, plain wall. SAME PERSON as the reference image (identity locked: {identity_lock(project, cid)}). "
                      f"{hdr}. Costume '{cost['label']}': {costume_phrase(cost)}. Used in scenes {', '.join(c['scenes'])}.",
            "negative": NEGATIVE, "palette": cost["colour_hex"], "traces": {"source_scenes": c["scenes"], "plan": ["costumes", k]}}
    for sc in st["scenes"]:
        a, loc = project.adaptation[sc["scene_id"]], pl["locations"][sc["location_id"]]
        who = sc["characters"]
        blocking = " ".join(b["en"] for b in a["beats"] if b["type"] == "action")[:600]
        costs = "; ".join(f"{pl['names'][c]['short']} wears '{pl['costumes'][sc['costume_ids'][c]]['label']}' ({pl['costumes'][sc['costume_ids'][c]]['garments']})" for c in who)
        props = ", ".join(pl["props"][p["prop_id"]]["adapted_name"] for p in sc["props_in"])
        deps = [f"IMG_CHAR_{c}" for c in who] + sorted({f"IMG_COST_{sc['costume_ids'][c]}" for c in who})
        specs[f"IMG_SCENE_{sc['scene_id']}"] = {"kind": "scene", "ref_id": sc["scene_id"], "model": model, "size": "1536x1024", "seed": seed, "deps": deps,
            "prompt": f"{STYLE}. Film keyframe, wide-medium shot. {hdr}. Location: {loc['description']}; architecture: {loc['architecture']}; dressing: {', '.join(loc['dressing'])}. "
                      f"Time: {sc['time']} - {TIME_LIGHT.get(sc['time'], 'natural light')}. Weather: {sc.get('weather') or 'clear'}. Mood: {sc['mood']}. "
                      f"Characters (keep faces/bodies/costumes IDENTICAL to the reference images): {costs}. Props: {props}. Blocking: {blocking}",
            "negative": NEGATIVE, "palette": loc["palette_hex"], "traces": {"source_scenes": [sc["scene_id"]], "plan": ["locations", sc["location_id"]],
                                                                              "adapted_scene": sc["scene_id"], "decisions": sorted({r for b in a["beats"] for r in b.get("plan_refs", [])})}}
        specs[f"IMG_SCENE_{sc['scene_id']}"]["adapted_hash"] = sha(a["beats"])
    return specs

def effective_hash(specs, aid, memo=None):
    memo = {} if memo is None else memo
    if aid not in memo:
        s = specs[aid]
        memo[aid] = sha({"spec": {k: v for k, v in s.items() if k not in ("traces",)}, "deps": [effective_hash(specs, d, memo) for d in s["deps"]]})
    return memo[aid]

def asset_status(project, specs=None):
    specs = specs or build_specs(project); memo, rows = {}, []
    for aid, s in specs.items():
        rec = project.assets.get(aid)
        eh = effective_hash(specs, aid, memo)
        status = "missing" if not rec else rec["status"]
        if rec and rec["status"] == "done" and rec["built_hash"] != eh: status = "stale"
        rows.append({"asset": aid, "kind": s["kind"], "status": status, "attempts": (rec or {}).get("attempts", 0), "error": (rec or {}).get("error", "")})
    return pd.DataFrame(rows)

### 10.2 Image backends (pluggable)
* **`openai`** – `gpt-image-1`: character = `generate`; costume & scene = `edit` with **reference images** (identity/costume lock).
* **`placeholder`** – PIL renders a deterministic card (palette + silhouettes + IDs + prompt snippet) — pipeline/tests offline chalte hain, aur woh clearly *placeholder* hai (real culture-recognition ke liye `openai` backend ya koi aur model chahiye).


In [ ]:
from PIL import Image, ImageDraw, ImageFont

def _hex(h, d=(90, 90, 90)):
    try: h = h.lstrip("#"); return tuple(int(h[i:i + 2], 16) for i in (0, 2, 4))
    except Exception: return d

def _wrap(draw, text, width_px, font):
    words, lines, cur = text.split(), [], ""
    for w in words:
        t = (cur + " " + w).strip()
        if draw.textlength(t, font=font) <= width_px: cur = t
        else: lines.append(cur); cur = w
    return lines + ([cur] if cur else [])

class PlaceholderBackend:
    name = "placeholder"
    def generate(self, spec, ref_paths, out_path):
        W, H = map(int, spec["size"].split("x")); W, H = W // 2, H // 2
        pal = [_hex(h) for h in (spec.get("palette") or ["#888888", "#555555", "#333333"])] or [(120, 120, 120)]
        img = Image.new("RGB", (W, H)); d = ImageDraw.Draw(img); f = ImageFont.load_default()
        top, bot = pal[0], pal[-1]
        for y in range(H):
            t = y / H; d.line([(0, y), (W, y)], fill=tuple(int(top[i] * (1 - t) + bot[i] * t * 0.6) for i in range(3)))
        if spec["kind"] in ("character", "costume"):
            cx = W // 2; body = pal[min(1, len(pal) - 1)]; trim = pal[-1]
            d.ellipse([cx - 40, 80, cx + 40, 160], fill=(205, 160, 120)); d.rectangle([cx - 45, 70, cx + 45, 105], fill=pal[-1] if len(pal) > 1 else (30, 30, 30))
            d.rectangle([cx - 60, 165, cx + 60, 380], fill=body); d.rectangle([cx - 45, 380, cx + 45, 560], fill=trim)
            d.rectangle([cx - 50, 560, cx - 5, 585], fill=(20, 20, 20)); d.rectangle([cx + 5, 560, cx + 50, 585], fill=(20, 20, 20))
        else:
            d.rectangle([0, int(H * 0.62), W, H], fill=tuple(max(0, c - 40) for c in pal[0]))
            n = max(1, len(spec["deps"]) // 2); 
            for i in range(n):
                x = int(W * (i + 1) / (n + 1)); c = pal[(i + 1) % len(pal)]
                d.ellipse([x - 18, int(H * 0.38), x + 18, int(H * 0.38) + 36], fill=(205, 160, 120)); d.rectangle([x - 28, int(H * 0.38) + 38, x + 28, int(H * 0.8)], fill=c)
        d.rectangle([0, H - 150, W, H], fill=(0, 0, 0))
        d.text((8, H - 144), f"[PLACEHOLDER] {spec['kind'].upper()} {spec['ref_id']}  seed={spec['seed']}", fill=(255, 220, 120), font=f)
        for i, ln in enumerate(_wrap(d, spec["prompt"], W - 16, f)[:10]): d.text((8, H - 128 + i * 12), ln, fill=(235, 235, 235), font=f)
        if ref_paths: d.text((8, 6), "refs: " + ", ".join(Path(p).stem for p in ref_paths)[:90], fill=(255, 255, 255), font=f)
        img.save(out_path)

class OpenAIBackend:
    name = "openai"
    def generate(self, spec, ref_paths, out_path):
        from openai import OpenAI
        c = OpenAI(); prompt = spec["prompt"] + " AVOID: " + spec["negative"]
        size = "1024x1536" if spec["size"].startswith("1024") else "1536x1024"
        if ref_paths:
            r = c.images.edit(model=CFG["openai_image_model"], image=[open(p, "rb") for p in ref_paths], prompt=prompt, size=size)
        else:
            r = c.images.generate(model=CFG["openai_image_model"], prompt=prompt, size=size)
        Path(out_path).write_bytes(base64.b64decode(r.data[0].b64_json))

def get_backend():
    return OpenAIBackend() if CFG["image_backend"] == "openai" else PlaceholderBackend()

### 10.3 Generation engine — gated, dependency-ordered, **retry one asset**, failures isolated
* Gate check: `extraction + plan + costumes` approved **aur** koi blocking continuity error nahi.
* Ek asset fail → status `failed`; uske dependents `blocked`; baaki chalte rehte hain.
* `retry_asset("IMG_SCENE_SC03")` sirf wahi dobara banata hai. Stale (hash-mismatch) assets hi regenerate hote hain.


In [ ]:
def _generate_one(project, aid, specs, backend, memo):
    s = specs[aid]; rec = project.assets.setdefault(aid, {"attempts": 0, "status": "pending"})
    blocked = [d for d in s["deps"] if project.assets.get(d, {}).get("status") != "done"]
    if blocked:
        rec.update(status="blocked", error=f"waiting on {blocked}"); return
    refs = [project.assets[d]["path"] for d in s["deps"]]
    out = project.dir / "images" / f"{aid}.png"
    for attempt in range(CFG["max_asset_attempts"]):
        rec["attempts"] += 1
        try:
            backend.generate(s, refs, out)
            rec.update(status="done", path=str(out), built_hash=effective_hash(specs, aid, memo), spec=s, backend=backend.name, error="", ts=utcnow())
            log_ai(f"image:{aid}", backend=backend.name, ok=True, model=s["model"]); return
        except Exception as e:
            rec.update(status="failed", error=str(e)[:300]); log_ai(f"image:{aid}", backend=backend.name, ok=False, error=str(e)[:200])
    project.failures.append({"stage": "image", "asset": aid, "error": rec["error"]})

def generate_assets(project, only=None, force=False):
    require(project, "extraction", "plan", "costumes")
    blocking = [i for i in project.issues if i["severity"] == "error" and i["code"] not in project.approvals["extraction"].get("waived", [])]
    if blocking: raise ApprovalBlocked(f"unresolved continuity errors: {[i['id'] for i in blocking]}")
    adv = [i for i in verify_adaptation(project) if i["severity"] == "error"]
    if adv: raise ApprovalBlocked(f"adaptation has errors: {[i['code'] for i in adv]}")
    specs, backend, memo, built = build_specs(project), get_backend(), {}, []
    for aid in specs:                                               # insertion order = chars -> costumes -> scenes
        rec = project.assets.get(aid)
        eh = effective_hash(specs, aid, memo)
        stale = (not rec) or rec["status"] != "done" or rec.get("built_hash") != eh
        if (only and aid not in only) or not (stale or force): continue
        _generate_one(project, aid, specs, backend, memo); built.append(aid)
    project.save()
    return built

def retry_asset(project, aid):
    return generate_assets(project, only=[aid], force=True)

### 10.4 Driver — generate the full pack once


In [ ]:
t0 = time.time()
built = generate_assets(P)
print(f"generated {len(built)} assets in {time.time()-t0:.1f}s with backend '{CFG['image_backend']}'")
display(asset_status(P))

### 10.5 Image verification (duplicates · references · optional vision judge)
* Structural: ek character = ek image; ek costume = ek image; har scene = ek keyframe; sab files exist; sab scene images ke deps approved assets hain.
* **Vision judge (live mode):** Claude (vision) scene keyframe ko character refs se compare karta hai (face/costume/culture). Mock mode mein skip.


In [ ]:
def verify_images(project):
    st, iss = project.state, []
    specs = build_specs(project); memo = {}
    for aid in specs:
        rec = project.assets.get(aid)
        if not rec or rec["status"] != "done": iss.append(issue("IMG_NOT_READY", "error", f"{aid} is {rec['status'] if rec else 'missing'}", entities=[aid])); continue
        if not Path(rec["path"]).exists(): iss.append(issue("IMG_FILE_MISSING", "error", f"{aid} file missing", entities=[aid]))
        if rec["built_hash"] != effective_hash(specs, aid, memo): iss.append(issue("IMG_STALE", "warning", f"{aid} is stale (upstream changed)", entities=[aid]))
    for kind, reg, pre in (("character", st["chars"], "IMG_CHAR_"), ("costume", st["costs"], "IMG_COST_")):
        for eid in reg:
            n = sum(1 for a in project.assets if a == pre + eid)
            if n != 1: iss.append(issue("IMG_DUPLICATE", "error", f"{kind} {eid} has {n} reference images (expected exactly 1)", entities=[eid]))
    for sc in st["scenes"]:
        need = {f"IMG_COST_{sc['costume_ids'][c]}" for c in sc["characters"]} | {f"IMG_CHAR_{c}" for c in sc["characters"]}
        if not need <= set(specs[f"IMG_SCENE_{sc['scene_id']}"]["deps"]): iss.append(issue("IMG_REF_MISSING", "error", f"{sc['scene_id']} keyframe does not reference all approved char/costume images", [sc["scene_id"]]))
    return iss

def vision_consistency_check(project, scene_id):
    """LIVE only: ask Claude (vision) whether a keyframe matches the approved references. Returns JSON verdict."""
    if MOCK_LLM: return {"skipped": "mock mode (needs ANTHROPIC_API_KEY)"}
    import anthropic
    aid = f"IMG_SCENE_{scene_id}"; spec = project.assets[aid]["spec"]
    content = []
    for d in spec["deps"] + [aid]:
        b64 = base64.b64encode(Path(project.assets[d]["path"]).read_bytes()).decode()
        content += [{"type": "text", "text": f"Image {d}:"}, {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": b64}}]
    content.append({"type": "text", "text": ("The last image is a scene keyframe; the earlier ones are approved character/costume references. Return ONLY JSON: "
        '{"faces_match": bool, "costumes_match": bool, "culture_recognisable": bool, "problems": [str]}. Selected culture: ' + culture_header(project.selection))})
    r = anthropic.Anthropic().messages.create(model=CFG["llm_model"], max_tokens=600, messages=[{"role": "user", "content": content}])
    log_ai(f"vision_check:{scene_id}", model=CFG["llm_model"], ok=True)
    return parse_json_loose("".join(b.text for b in r.content if b.type == "text"))

display(issues_table(verify_images(P)) if verify_images(P) else pd.DataFrame([{"result": "image verification: no issues"}]))
print(vision_consistency_check(P, "SC03"))

### 10.6 Gallery (by character / costume / scene, linked to canonical IDs)


In [ ]:
def gallery(project, kind="all", width=240):
    from IPython.display import HTML
    cards = []
    for aid, rec in project.assets.items():
        if rec.get("status") != "done" or (kind != "all" and rec["spec"]["kind"] != kind): continue
        b64 = base64.b64encode(Path(rec["path"]).read_bytes()).decode()
        refs = rec["spec"]["traces"].get("source_scenes", [])
        cards.append(f"<div style='display:inline-block;margin:6px;width:{width}px;vertical-align:top;font:12px sans-serif'>"
                     f"<img src='data:image/png;base64,{b64}' style='width:100%;border-radius:6px'>"
                     f"<b>{html.escape(aid)}</b><br>kind: {rec['spec']['kind']} - scenes: {html.escape(', '.join(refs))}<br>backend: {rec['backend']}</div>")
    return HTML("".join(cards) or "<i>no images yet</i>")

gallery(P, "character")

In [ ]:
gallery(P, "costume")

In [ ]:
gallery(P, "scene", width=360)

---
# STEP 11 — Correct a choice and regenerate **only what is affected**
Assignment demo: *"user corrects a cultural choice and regenerates only affected outputs."* Yahan hum Jeeta ke kapde ke rang ka plan edit karte hain → approvals invalidate → re-approve → sirf **COST_AMAR_02** aur uss costume wale **scene images (SC03, SC04)** stale/regenerate hote hain.


In [ ]:
before = asset_status(P).set_index("asset")["status"].to_dict()

edit_plan(P, ["costumes", "COST_AMAR_02", "colours"], "cream-white, black", reason="Reviewer: pure white is for mourning-style dress; prefer cream-white kurta")
edit_plan(P, ["costumes", "COST_AMAR_02", "colour_hex"], ["#efe6cf", "#111111"])
print("approvals after edit ->"); display(approval_status(P))      # plan/costumes invalidated by hash
approve(P, "plan"); approve(P, "costumes")

stale = asset_status(P); print("\nStale / affected assets:"); display(stale[stale.status != "done"])
rebuilt = generate_assets(P)
print("\nRegenerated ONLY:", rebuilt)
assert set(rebuilt) == {"IMG_COST_COST_AMAR_02", "IMG_SCENE_SC03", "IMG_SCENE_SC04"}, rebuilt

# retry a single (e.g. failed) asset
print("single retry ->", retry_asset(P, "IMG_SCENE_SC02"))

---
# STEP 12 — Export package (structured data + readable reports + images)
Output folder `sample_output/<culture>/` assignment ke submission structure ko follow karta hai: `scene_breakdown.json`, adapted screenplay (MD + HTML [+ PDF agar `weasyprint` ho]), continuity report, character/costume bibles, scene keyframes, trace file, AI-usage log, aur `.zip`.


In [ ]:
def esc(x): return html.escape(str(x))

def report_html(title, body): 
    return (f"<!doctype html><meta charset='utf-8'><title>{esc(title)}</title><style>body{{font:15px/1.5 system-ui,sans-serif;max-width:900px;margin:2em auto;padding:0 1em}}"
            f"table{{border-collapse:collapse;width:100%}}td,th{{border:1px solid #ccc;padding:4px 8px;vertical-align:top}}.error{{color:#b00020}}.warning{{color:#b26a00}}</style><h1>{esc(title)}</h1>{body}")

def continuity_report_html(project):
    rows = "".join(f"<tr class='{i['severity']}'><td>{i['id']}</td><td>{i['severity']}</td><td>{i['code']}</td><td>{esc(', '.join(i['scenes']))}</td><td>{esc(i['message'])}</td></tr>" for i in project.issues)
    fixes = "".join(f"<li>{esc(e['ts'])} - {e['action']}: {esc(e.get('reason') or e.get('fields') or e.get('keep') or '')}</li>" for e in project.edit_log if e["action"] in ("patch_scene", "merge", "edit_entity", "merge_costume"))
    ap = "".join(f"<li>{g}: approved by {a['by']} at {a['ts']} (hash {a['hash']}) - currently {'VALID' if is_approved(project, g) else 'INVALIDATED'}</li>" for g, a in project.approvals.items())
    return report_html("Continuity report", f"<p>Open issues after review: <b>{len(project.issues)}</b></p><table><tr><th>ID</th><th>Severity</th><th>Code</th><th>Scenes</th><th>Message</th></tr>{rows}</table><h2>Corrections made (audit trail)</h2><ul>{fixes}</ul><h2>Approvals</h2><ul>{ap}</ul>")

def screenplay_html(project):
    out = []
    for sid, a in sorted(project.adaptation.items()):
        out.append(f"<h2>{sid}: {esc(a['heading'])} <small>({esc(a['heading_roman'])})</small></h2>")
        for b in a["beats"]:
            if b["type"] == "action": out.append(f"<p><i>{esc(b['text'])}</i><br><small>{esc(b['en'])} - {esc(b['plan_refs'])}</small></p>")
            else: out.append(f"<p><b>{esc(b['speaker'])}</b> {esc('('+b['paren']+')' if b['paren'] else '')}: {esc(b['line'])}<br><small>{esc(b['roman'])} - &ldquo;{esc(b['en'])}&rdquo; - {esc(b['plan_refs'])}</small></p>")
        out.append("<h4>Why it changed</h4><table><tr><th>Source</th><th>Adapted</th><th>Why</th><th>Plan refs</th></tr>" + "".join(f"<tr><td>{esc(c['source'])}</td><td>{esc(c['adapted'])}</td><td>{esc(c['why'])}</td><td>{esc(c['plan_refs'])}</td></tr>" for c in a["changes"]) + "</table>")
    return report_html(f"Adapted screenplay - {project.selection['dialect']} ({project.selection['script']})", "".join(out))

def bible_md(project, kind):
    pl, st, L = project.plan, project.state, []
    if kind == "character":
        for cid, c in st["chars"].items():
            ch, nm = pl["characters"][cid], pl["names"][cid]
            L += [f"## {cid} - {nm['adapted_name']}", f"- source name: {c['name']} (aliases: {', '.join(c['aliases'])}); role: {c.get('role')}; age {c.get('age')}",
                  f"- physical: {ch['physical']}", f"- grooming: {ch['grooming']}", f"- personality cues: {ch['personality_cues']}", f"- speech: {ch['speech']}; gestures: {ch['gesture_habits']}",
                  f"- scenes: {', '.join(c['scenes'])}; reference image: IMG_CHAR_{cid}.png\n"]
    else:
        for k, c in st["costs"].items():
            cs = pl["costumes"][k]
            L += [f"## {k} - {cs['label']} ({c['char_id']})", f"- garments: {cs['garments']}", f"- fabrics: {cs['fabrics']}; colours: {cs['colours']}", f"- footwear: {cs['footwear']}; jewellery: {cs['jewellery']}; headwear: {cs['headwear']}; grooming: {cs['grooming']}",
                  f"- scenes worn: {', '.join(c['scenes'])}; change reason: {cs['change_reason'] or '-'}; image: IMG_COST_{k}.png\n"]
    return "\n".join(L)

def export_pack(project, out_root="sample_output"):
    require(project, "extraction", "plan", "costumes")
    root = Path(out_root) / project.slug
    if root.exists(): shutil.rmtree(root)
    for sub in ("character_bible", "costume_bible", "scene_keyframes", "specs"): (root / sub).mkdir(parents=True)
    st = project.state
    (root / "scene_breakdown.json").write_text(json.dumps({k: st[k] for k in ("scenes", "chars", "locs", "props", "costs")}, ensure_ascii=False, indent=2), encoding="utf-8")
    (root / "adaptation_plan.json").write_text(json.dumps(project.plan, ensure_ascii=False, indent=2), encoding="utf-8")
    (root / "adapted_screenplay.json").write_text(json.dumps(project.adaptation, ensure_ascii=False, indent=2), encoding="utf-8")
    (root / "adapted_screenplay.md").write_text(screenplay_markdown(project), encoding="utf-8")
    (root / "adapted_screenplay.html").write_text(screenplay_html(project), encoding="utf-8")
    (root / "continuity_report.html").write_text(continuity_report_html(project), encoding="utf-8")
    (root / "continuity_report.json").write_text(json.dumps({"issues": project.issues, "edit_log": project.edit_log, "resolution_log": project.res_log}, ensure_ascii=False, indent=2), encoding="utf-8")
    (root / "character_bible" / "character_bible.md").write_text(bible_md(project, "character"), encoding="utf-8")
    (root / "costume_bible" / "costume_bible.md").write_text(bible_md(project, "costume"), encoding="utf-8")
    for aid, rec in project.assets.items():
        if rec.get("status") != "done": continue
        folder = {"character": "character_bible", "costume": "costume_bible", "scene": "scene_keyframes"}[rec["spec"]["kind"]]
        shutil.copy(rec["path"], root / folder / f"{aid}.png")
        (root / "specs" / f"{aid}.json").write_text(json.dumps({"asset": aid, **{k: v for k, v in rec["spec"].items()}, "built_hash": rec["built_hash"], "backend": rec["backend"]}, ensure_ascii=False, indent=2), encoding="utf-8")
    if USAGE_LOG.exists(): shutil.copy(USAGE_LOG, root / "ai_usage_log.jsonl")
    try:                                                          # optional PDFs (needs weasyprint + a Gurmukhi/Devanagari font on the system)
        from weasyprint import HTML as WHTML
        WHTML(string=(root / "adapted_screenplay.html").read_text(encoding="utf-8")).write_pdf(root / "adapted_screenplay.pdf")
        WHTML(string=(root / "continuity_report.html").read_text(encoding="utf-8")).write_pdf(root / "continuity_report.pdf")
    except Exception as e:
        (root / "PDF_NOTE.txt").write_text(f"PDF skipped ({type(e).__name__}); open the .html files and print to PDF.", encoding="utf-8")
    z = shutil.make_archive(str(root), "zip", root)
    return root, Path(z)

root, zip_path = export_pack(P)
print("exported to", root, "->", zip_path, f"({zip_path.stat().st_size/1024:.0f} KB)")
for p in sorted(root.rglob("*")):
    if p.is_file(): print("  ", p.relative_to(root))

---
# STEP 13 — Interactive Studio Dashboard (ipywidgets · `voila` se web app)

This dashboard calls the same functions created in the steps above — **no separate logic**, so the notebook and UI remain consistent.
Tabs: **Setup → Scenes → Review & Approve → Plan → Screenplay → Gallery → Continuity & Export**.

Web app ki tarah chalane ke liye: `pip install voila && voila Cultural_Adaptation_Studio.ipynb`


In [ ]:
APP = {"p": P}      # dashboard works on whichever project is active

def build_dashboard():
    import ipywidgets as W
    from IPython.display import display, HTML, clear_output

    def box(): return W.Output(layout=W.Layout(border="1px solid #ddd", padding="6px", max_height="520px", overflow="auto"))
    def run(out, fn):
        with out:
            clear_output(wait=True)
            try: fn()
            except Exception as e: print(f"[{type(e).__name__}] {e}")
    def btn(label, style="primary"): return W.Button(description=label, button_style={"primary": "primary", "ok": "success", "warn": "warning"}.get(style, ""), layout=W.Layout(width="auto"))
    p = lambda: APP["p"]

    # ---- 1 Setup -----------------------------------------------------------------
    dialect = W.Text(value=SELECTION["dialect"], description="Dialect"); region = W.Text(value=SELECTION["region"], description="Region", layout=W.Layout(width="600px"))
    culture = W.Text(value=SELECTION["culture"], description="Culture")
    setting = W.Dropdown(options=["rural", "urban"], value=SELECTION["setting"], description="Setting")
    script = W.Dropdown(options=list(SCRIPTS), value=SELECTION["script"], description="Script")
    text = W.Textarea(value=SAMPLE_SCREENPLAY, description="Screenplay", layout=W.Layout(width="95%", height="180px"))
    o1, b1 = box(), btn("1) Create project -> extract -> normalize -> check", "ok")
    def go_setup(_):
        def f():
            sel = dict(SELECTION, dialect=dialect.value, region=region.value, culture=culture.value, setting=setting.value, script=script.value)
            APP["p"] = Project(sel); ingest(APP["p"], text.value); run_extraction(APP["p"]); refresh(APP["p"])
            print("project:", APP["p"].slug, "| scenes:", len(APP["p"].state["scenes"]), "| issues:", len(APP["p"].issues))
            for pr in APP["p"].ingest_problems: print(f"  [{pr['severity']}] {pr['code']}: {pr['message']}")
        run(o1, f)
    b1.on_click(go_setup)

    # ---- 2 Scenes + inline fix ------------------------------------------------------
    o2, b2 = box(), btn("Refresh tables")
    def show_scenes(_=None):
        def f():
            display(scene_table(p().state)); display(char_table(p().state)); display(costume_table(p().state))
        run(o2, f)
    b2.on_click(show_scenes)
    sc_no = W.BoundedIntText(value=2, min=1, max=9, description="Scene #")
    fld = W.Dropdown(options=["transfers", "props_in", "costume_change_reason", "costumes", "characters", "time", "summary"], description="Field")
    val = W.Textarea(placeholder='JSON value, e.g. {"The Elder Son": "bathed and changed"}', layout=W.Layout(width="95%", height="70px"))
    why = W.Text(placeholder="reason (logged)", layout=W.Layout(width="60%"))
    o2b, b2b = box(), btn("Apply patch", "warn")
    def do_patch(_):
        def f():
            patch_scene(p(), sc_no.value, fld.value, json.loads(val.value), why.value or "dashboard edit")
            print("patched. open issues:", len(p().issues)); display(issues_table(p().issues))
        run(o2b, f)
    b2b.on_click(do_patch)

    # ---- 3 Review & approve --------------------------------------------------------------
    keep, drop = W.Text(placeholder="keep ID e.g. CHAR_AMAR", description="Keep"), W.Text(placeholder="drop ID", description="Drop")
    newname = W.Text(placeholder="ID=New Name", description="Rename")
    waive = W.Text(placeholder="waive codes (comma sep, optional)", description="Waive")
    o3 = box(); bm, br, ba = btn("Merge", "warn"), btn("Rename"), btn("Approve extraction", "ok")
    def f_merge(_): run(o3, lambda: (merge_entities(p(), keep.value.strip(), drop.value.strip()), display(char_table(p().state)), display(issues_table(p().issues))))
    def f_rename(_): run(o3, lambda: (lambda i, n: (rename_entity(p(), i.strip(), n.strip()), display(char_table(p().state))))(*newname.value.split("=", 1)))
    def f_appr(_): run(o3, lambda: (approve(p(), "extraction", waive=[w.strip() for w in waive.value.split(",") if w.strip()]), display(approval_status(p()))))
    bm.on_click(f_merge); br.on_click(f_rename); ba.on_click(f_appr)

    # ---- 4 Plan ---------------------------------------------------------------------------------
    o4 = box(); bp, bpa, bpc = btn("Build cultural plan"), btn("Approve plan", "ok"), btn("Approve costume plan", "ok")
    did = W.Text(placeholder="D06", description="Decision"); dst = W.Dropdown(options=["proposed", "accepted", "edited", "rejected"], description="Status"); bds = btn("Set status")
    def f_plan(_): run(o4, lambda: (build_plan(p()), display(plan_table(p())), display(pd.DataFrame(p().plan["uncertainties"]))))
    bp.on_click(f_plan)
    bpa.on_click(lambda _: run(o4, lambda: (approve(p(), "plan"), display(approval_status(p())))))
    bpc.on_click(lambda _: run(o4, lambda: (approve(p(), "costumes"), display(approval_status(p())))))
    bds.on_click(lambda _: run(o4, lambda: (set_decision_status(p(), did.value.strip(), dst.value), display(plan_table(p())))))

    # ---- 5 Screenplay comparison -----------------------------------------------------------------
    o5, b5 = box(), btn("Adapt screenplay + verify")
    pick = W.Dropdown(options=["SC01", "SC02", "SC03", "SC04"], description="Scene"); b5s = btn("Show source vs adapted")
    def f_adapt(_): run(o5, lambda: (adapt_screenplay(p()), display(issues_table(verify_adaptation(p())))))
    def f_cmp(_):
        def f():
            sid = pick.value; s = next(x for x in p().state["scenes"] if x["scene_id"] == sid); a = p().adaptation[sid]
            left = f"<pre style='white-space:pre-wrap'>{esc(s['source_text'])}</pre>"
            right = "".join((f"<p><b>{esc(b['speaker'])}</b>: {esc(b['line'])}<br><small>{esc(b['roman'])} - {esc(b['en'])} {esc(b['plan_refs'])}</small></p>" if b["type"] == "dialogue" else f"<p><i>{esc(b['text'])}</i><br><small>{esc(b['en'])}</small></p>") for b in a["beats"])
            why = "".join(f"<li><b>{esc(c['source'])}</b> -> {esc(c['adapted'])}: {esc(c['why'])} {esc(c['plan_refs'])}</li>" for c in a["changes"])
            display(HTML(f"<table width='100%'><tr><th>Source {sid}</th><th>Adapted - {esc(a['heading'])}</th></tr><tr><td valign='top' width='40%'>{left}</td><td valign='top'>{right}</td></tr></table><h4>Why it changed</h4><ul>{why}</ul>"))
        run(o5, f)
    b5.on_click(f_adapt); b5s.on_click(f_cmp)

    # ---- 6 Gallery ------------------------------------------------------------------------------
    o6, bg = box(), btn("Generate / regenerate stale assets", "ok")
    kind = W.Dropdown(options=["all", "character", "costume", "scene"], description="Show"); bgs = btn("Show gallery")
    rid = W.Text(placeholder="IMG_SCENE_SC03", description="Retry asset"); brr = btn("Retry this asset only", "warn")
    bg.on_click(lambda _: run(o6, lambda: (print("built:", generate_assets(p())), display(asset_status(p())))))
    bgs.on_click(lambda _: run(o6, lambda: display(gallery(p(), kind.value))))
    brr.on_click(lambda _: run(o6, lambda: (print("rebuilt:", retry_asset(p(), rid.value.strip())), display(gallery(p(), "all")))))

    # ---- 7 Continuity & export -----------------------------------------------------------------------
    o7, bc, be = box(), btn("Re-run all checks"), btn("Export package", "ok")
    bc.on_click(lambda _: run(o7, lambda: (refresh(p()), display(issues_table(p().issues)), display(approval_status(p())), display(asset_status(p())) if is_approved(p(), "costumes") else None)))
    be.on_click(lambda _: run(o7, lambda: print("exported ->", export_pack(p())[1])))

    tabs = W.Tab(children=[
        W.VBox([culture, dialect, region, W.HBox([setting, script]), text, b1, o1]),
        W.VBox([b2, o2, W.HTML("<b>Fix an extraction error</b>"), W.HBox([sc_no, fld]), val, why, b2b, o2b]),
        W.VBox([W.HBox([keep, drop, bm]), W.HBox([newname, br]), W.HBox([waive, ba]), o3]),
        W.VBox([W.HBox([bp, bpa, bpc]), W.HBox([did, dst, bds]), o4]),
        W.VBox([b5, W.HBox([pick, b5s]), o5]),
        W.VBox([bg, W.HBox([kind, bgs]), W.HBox([rid, brr]), o6]),
        W.VBox([W.HBox([bc, be]), o7])])
    for i, t in enumerate(["1 Setup", "2 Scenes", "3 Review & Approve", "4 Plan", "5 Screenplay", "6 Gallery", "7 Continuity & Export"]): tabs.set_title(i, t)
    return tabs

try:
    dashboard = build_dashboard(); display(dashboard)
except ImportError as e:
    print("ipywidgets not installed:", e)

---
# STEP 14 — Automated tests (`unittest`, inline)
Har test ek **fresh Project** (alag folder) par chalta hai. Covered: alias merge, duplicate costumes, contradiction detection, approval gates, hash-invalidation, selective regeneration, ambiguity, culture isolation, missing scenes, script check, broad-label rejection.


In [ ]:
import unittest

def fresh_project(dialect="Majhi", region="Test region", approve_all=False):
    sel = dict(SELECTION, dialect=dialect, region=region)
    q = Project(sel); ingest(q, SAMPLE_SCREENPLAY); run_extraction(q); refresh(q)
    if approve_all:
        patch_scene(q, 2, "transfers", q.raw["scenes"][1]["transfers"] + [{"prop": "letter", "from": "Karan", "to": "Amar"}], "test fix")
        patch_scene(q, 3, "costume_change_reason", {"The Elder Son": "bathed"}, "test fix")
        approve(q, "extraction"); build_plan(q)
        for d in q.plan["decisions"]: d["status"] = "accepted"
        approve(q, "plan"); approve(q, "costumes"); adapt_screenplay(q)
    return q

class StudioTests(unittest.TestCase):
    def test_scene_split_order_and_problems(self):
        sc, _, pr = split_scenes(SAMPLE_SCREENPLAY)
        self.assertEqual([s["number"] for s in sc], [1, 2, 3, 4])
        self.assertEqual(sc[0]["time"], "night")
        _, _, pr2 = split_scenes("just some text without any sluglines")
        self.assertIn("NO_SLUGLINES", [x["code"] for x in pr2])

    def test_alias_merge_single_canonical_character(self):
        q = fresh_project()
        self.assertEqual(len(q.state["chars"]), 4)
        self.assertEqual(q.state["chars"]["CHAR_AMAR"]["scenes"], ["SC01", "SC02", "SC03", "SC04"])
        self.assertTrue({"the elder son", "a. singh"} <= {a.lower() for a in q.state["chars"]["CHAR_AMAR"]["aliases"]})

    def test_costumes_deduplicated(self):
        q = fresh_project()
        self.assertEqual(sorted(q.state["costs"]), ["COST_AMAR_01", "COST_AMAR_02", "COST_FATHER_01", "COST_KARAN_01", "COST_MOTHER_01"])
        self.assertEqual(q.state["costs"]["COST_FATHER_01"]["scenes"], ["SC01", "SC03", "SC04"])   # Same costume reused across 3 scenes
        self.assertFalse([i for i in q.issues if i["code"].startswith("DUPLICATE")])

    def test_contradictions_detected_with_affected_scenes(self):
        q = fresh_project()
        tele = [i for i in q.issues if i["code"] == "PROP_TELEPORT"][0]
        self.assertEqual(tele["scenes"], ["SC02", "SC03"])
        cc = [i for i in q.issues if i["code"] == "COSTUME_CHANGE_UNEXPLAINED"][0]
        self.assertEqual(cc["scenes"], ["SC02", "SC03"])

    def test_knowledge_before_gain_detected(self):
        q = fresh_project(); q.raw["scenes"][3]["knowledge_use"].append({"char": "Karan", "fact": "Amar quit in anger"}); refresh(q)
        self.assertIn("KNOWLEDGE_BEFORE_GAIN", [i["code"] for i in q.issues])

    def test_missing_scene_and_unknown_speaker(self):
        q = fresh_project(); q.raw["scenes"].pop(); refresh(q)
        self.assertIn("MISSING_SCENE", [i["code"] for i in q.issues])

    def test_approval_blocked_until_errors_fixed(self):
        q = fresh_project()
        with self.assertRaises(ApprovalBlocked): approve(q, "extraction")

    def test_generation_blocked_without_approval(self):
        q = fresh_project()
        with self.assertRaises(ApprovalBlocked): generate_assets(q)
        with self.assertRaises(ApprovalBlocked): build_plan(q)

    def test_approval_invalidated_by_upstream_edit(self):
        q = fresh_project(approve_all=True)
        self.assertTrue(is_approved(q, "plan"))
        rename_entity(q, "CHAR_KARAN", "Karan Renamed")
        self.assertFalse(is_approved(q, "extraction")); self.assertFalse(is_approved(q, "plan")); self.assertFalse(is_approved(q, "costumes"))
        with self.assertRaises(ApprovalBlocked): generate_assets(q)

    def test_merge_entities_creates_one_record(self):
        q = fresh_project(); q.raw["characters"].append({"name": "Kuljeet", "aliases": [], "age": 30, "role": "x"}); refresh(q)
        self.assertIn("CHAR_KULJEET", q.state["chars"])
        merge_entities(q, "CHAR_KARAN", "CHAR_KULJEET")
        self.assertNotIn("CHAR_KULJEET", q.state["chars"])
        self.assertIn("kuljeet", {a.lower() for a in q.state["chars"]["CHAR_KARAN"]["aliases"]})

    def test_ambiguous_alias_is_flagged_not_guessed(self):
        q = fresh_project(); q.raw["characters"].append({"name": "Gurdev Singh", "aliases": [], "age": 40, "role": "uncle"})
        q.raw["scenes"][0]["characters"].append("Singh"); refresh(q)
        self.assertIn("AMBIGUOUS_ALIAS", [i["code"] for i in q.issues])

    def test_selective_regeneration_and_single_retry(self):
        q = fresh_project(approve_all=True)
        first = generate_assets(q); self.assertEqual(len(first), 4 + 5 + 4)
        self.assertEqual(generate_assets(q), [])                                   # nothing stale => nothing rebuilt
        edit_plan(q, ["costumes", "COST_KARAN_01", "colours"], "indigo, black"); approve(q, "costumes")
        # Karan's reference image wears his first costume, so it is correctly stale too (identity/costume stay in sync)
        self.assertEqual(generate_assets(q), ["IMG_CHAR_CHAR_KARAN", "IMG_COST_COST_KARAN_01", "IMG_SCENE_SC02", "IMG_SCENE_SC04"])
        self.assertEqual(retry_asset(q, "IMG_SCENE_SC01"), ["IMG_SCENE_SC01"])
        self.assertFalse(verify_images(q))

    def test_adaptation_verification_catches_dropped_prop_transfer(self):
        q = fresh_project(approve_all=True)
        self.assertFalse([i for i in verify_adaptation(q) if i["severity"] == "error"])
        q.adaptation["SC03"]["events"]["transfers"] = []
        self.assertIn("ADAPT_PLOT_HOLD".replace("HOLD", "HOLE"), [i["code"] for i in verify_adaptation(q)])

    def test_script_check(self):
        self.assertGreater(script_ratio("ਬਹਿ ਜਾ।", "Gurmukhi"), 0.9); self.assertLess(script_ratio("beh ja", "Gurmukhi"), 0.1)

    def test_broad_label_rejected(self):
        with self.assertRaises(ValueError): validate_selection(dict(SELECTION, dialect="Punjabi"))

    def test_culture_isolation(self):
        a = fresh_project(approve_all=True)
        b = Project(dict(SELECTION, culture="Rajasthani", dialect="Marwari", region="Marwar (Jodhpur), Rajasthan", script="Devanagari"))
        ingest(b, SAMPLE_SCREENPLAY); run_extraction(b); refresh(b)
        self.assertNotEqual(a.dir, b.dir)
        self.assertEqual(b.assets, {}); self.assertIsNone(b.plan); self.assertFalse(b.approvals)
        patch_scene(b, 2, "transfers", b.raw["scenes"][1]["transfers"] + [{"prop": "letter", "from": "Karan", "to": "Amar"}]); patch_scene(b, 3, "costume_change_reason", {"The Elder Son": "bathed"})
        approve(b, "extraction"); build_plan(b)
        self.assertNotIn("Bebe", plan_text(b.plan)); self.assertNotIn("Majhi", plan_text(b.plan))      # Nothing leaked from the Majhi project
        self.assertTrue(mixing_lint(a.selection, "ghoomar and bajra rotla"))

suite = unittest.defaultTestLoader.loadTestsFromTestCase(StudioTests)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "tests failed"

---
# STEP 15 — AI-usage log (traceability of every model call)
`ai_usage_log.jsonl` automatically har LLM / image call record karta hai. Assignment ke `AI_USAGE.md` mein **kya galat hua + kaise verify kiya** apne words mein add karna zaroori hai (neeche template).


In [ ]:
if USAGE_LOG.exists():
    log = pd.read_json(USAGE_LOG, lines=True)
    display(log.groupby(["mode" if "mode" in log else "purpose"]).size().rename("calls").to_frame() if "mode" in log else log.head())
    display(log.tail(8))

### Human log template (copy into `AI_USAGE.md`)

| Where AI was used | What it produced | What it got wrong | How I verified / fixed |
|---|---|---|---|
| Extraction (Claude) | scene records | *e.g. treated "A. Singh" as a new character* | `Resolver` token-subset + alias index; `check_duplicates`; unit test `test_alias_merge...` |
| Plan (Claude) | cultural decisions | *e.g. generic "turban" for every man* | uncertainty register + native-speaker review + `mixing_lint` |
| Rewrite (Claude) | Gurmukhi dialogue | *e.g. literal translation of "You've become thin"* | decision D01 forced idiom; verifier `ADAPT_WRONG_SCRIPT`, plot-hole guard |
| Images | keyframes | *e.g. face drift between scenes* | reference-image chaining + `vision_consistency_check` + hash staleness |
| Coding assistant | this notebook | *fill in honestly* | tests in Step 14 |


---
# STEP 16 — Write the submission documents (README · ARCHITECTURE · KNOWN_LIMITATIONS · AI_USAGE)


In [ ]:
SUB = Path("submission"); SUB.mkdir(exist_ok=True)
(SUB / "README.md").write_text("""# Cultural Screenplay & Visual Adaptation Studio

## Run
```bash
python -m venv .venv && source .venv/bin/activate
pip install anthropic openai pandas ipywidgets pillow python-docx pypdf jupyter voila weasyprint   # last one optional (PDF)
cp .env.example .env     # add ANTHROPIC_API_KEY (and OPENAI_API_KEY for real images)
jupyter lab Cultural_Adaptation_Studio.ipynb      # run all cells top to bottom
voila Cultural_Adaptation_Studio.ipynb            # or serve as a web app
```
No keys? The notebook runs in MOCK mode with a bundled 4-scene sample (Majhi / Gurmukhi) and placeholder image cards.

## Requirements
Python 3.10+, Jupyter, internet only for live LLM/image calls.

## Flow
Upload -> Select culture -> Extract -> Normalize -> Continuity checks -> **Gate 1** -> Plan -> **Gate 2/3** -> Rewrite -> Verify -> Generate -> Verify -> Export.
""")
(SUB / "ARCHITECTURE.md").write_text("""# Architecture
* **Project** = one culture = one folder; no shared state between cultures.
* **raw** LLM extraction + **overrides** (alias merges, edits) --normalize()--> canonical registries (CHAR_/LOC_/PROP_/COST_ IDs) + scene state.
* **Validators** (deterministic): extraction integrity, duplicates, props/costume/knowledge/injury continuity, plan coverage, adaptation (script, trace, prop transfers), images.
* **Gates**: content-hash approvals (extraction, plan, costumes); any upstream edit invalidates downstream approvals.
* **Assets**: spec (model, prompt, seed, refs) -> effective hash incl. dependency hashes -> only stale assets rebuild; per-asset retry; failures isolated.
* **Traceability**: adapted beats carry plan decision IDs + source scene; each image spec carries `traces`.
""")
(SUB / "KNOWN_LIMITATIONS.md").write_text("""# Known limitations
1. Bundled Majhi plan/dialogue are illustrative and NOT native-verified; live outputs need native-speaker review.
2. Cultural-mixing lint is a keyword heuristic - it misses subtle blending and can false-positive on shared Punjabi/Haryanvi vocabulary.
3. Placeholder images are not culturally meaningful; real recognisability depends on the image model.
4. Identity consistency relies on reference-image chaining; models can still drift (vision check is advisory).
5. Extraction accuracy depends on the LLM; scene splitting needs INT./EXT. sluglines (no OCR for scanned PDFs).
6. Costume dedupe uses token overlap (Jaccard>=0.6) - may merge distinct but similar outfits; reviewer can split by editing descriptions.
7. Notebook UI (ipywidgets) is an MVP; no auth, no multi-user, state is JSON on disk.
""")
(SUB / "AI_USAGE.md").write_text("# AI usage\nSee workspace/ai_usage_log.jsonl (auto-generated) and add the human-written table from the notebook (Step 15).\n")
print("written:", [p.name for p in SUB.iterdir()])

---
# STEP 17 — Live-demo checklist (assignment §14 → kahan dikhana hai)

| Demo item | Notebook cell / function |
|---|---|
| Character: source mention → canonical → adapted identity → visual | Step 5.3 (`aliases`), Step 8 (`names`, `characters`), Step 10 (`IMG_CHAR_CHAR_AMAR`) |
| Costume / prop across scenes | `costume_table` (COST_FATHER_01 in SC01, SC03, SC04), `PROP_LETTER` holder chain in Step 6.2 |
| Cultural decision beyond translation | Plan **D06** (seat on the manji / *paindi*) + **D07** (*pind wale ki kehnge*) |
| Contradiction caught before generation | Step 6.3 (`PROP_TELEPORT`, `COSTUME_CHANGE_UNEXPLAINED`) + Step 7.3 (approval **BLOCKED**) |
| Correct a choice → regenerate only affected | Step 11 (only `IMG_COST_COST_AMAR_02`, `IMG_SCENE_SC03`, `IMG_SCENE_SC04`) |
| AI usage + what it got wrong | Step 15 |

### Next steps (if time remains)
1. **Second culture** (bonus): naya `Project(...)` with e.g. Marwari/Devanagari — folders aur prompts alag; `test_culture_isolation` already proves no leakage.
2. Native-speaker review UI: decision-level comments + diff of re-written lines.
3. LLM "cultural judge" pass on the rewrite (second-opinion model) before showing to user.
4. Streamlit/FastAPI front end on the same functions for a standalone web deployment.
